In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping_update_calo.df"
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_muon_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 20)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
mc_bnb_hit0_df.index

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:
import pandas as pd

# Define the target 4 index levels identifying unique tracks/slices
target_levels = ['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']

# --- 1. Combined Unique Combinations Across hit0 + hit1 + hit2 ---
hit_dfs = {
    '0': mc_bnb_hit0_df,
    '1': mc_bnb_hit1_df,
    '2': mc_bnb_hit2_df,
}

hit_tuple_sets = []
total_hit_rows = 0

for name, df in hit_dfs.items():
    if not df.empty:
        total_hit_rows += len(df)
        # Extract target 4-tuples as a set
        tuples_set = set(
            df.index.to_frame()[target_levels].itertuples(index=False, name=None)
        )
        hit_tuple_sets.append(tuples_set)

# Union of unique 4-tuples across hit0, hit1, and hit2
if hit_tuple_sets:
    combined_hit_unique_tuples = set.union(*hit_tuple_sets)
    n_unique_combined_hits = len(combined_hit_unique_tuples)
else:
    n_unique_combined_hits = 0

print(f"Combined (hit0+hit1+hit2) total hit rows: {total_hit_rows}")
print(f"Combined (hit0+hit1+hit2) unique 4-tuple combinations: {n_unique_combined_hits}")

print("\n" + "=" * 50 + "\n")

# --- 2. Unique Combinations for mc_bnb_pfp_df ---
mc_bnb_pfp_df = mc_bnb_pfp_df.sort_index(level='__ntuple', ascending=True)

pfp_tuples_set = set(
    mc_bnb_pfp_df.index.to_frame()[target_levels].itertuples(index=False, name=None)
)
n_unique_pfp = len(pfp_tuples_set)

print(f"mc_bnb_pfp_df total rows: {len(mc_bnb_pfp_df)}")
print(f"mc_bnb_pfp_df unique 4-tuple combinations: {n_unique_pfp}")

# --- Optional Check: Overlap between Hits and PFP ---
if n_unique_combined_hits > 0 and n_unique_pfp > 0:
    overlap = len(combined_hit_unique_tuples.intersection(pfp_tuples_set))
    print("\n" + "=" * 50 + "\n")
    print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

In [ ]:
import pandas as pd

# Define your columns
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Optional: set to None if unweighted

# --- 1. Extract and Clean Data ---
valid_mask = mc_bnb_pfp_df[p_type_col].notna()
df_clean = mc_bnb_pfp_df[valid_mask]

if weight_col and weight_col in df_clean.columns:
    weights = df_clean[weight_col].fillna(1.0)
else:
    weights = pd.Series(1.0, index=df_clean.index)

# --- 2. Calculate Weighted & Unweighted Statistics ---
stats_df = pd.DataFrame({
    'p_type': df_clean[p_type_col],
    'weight': weights
})

summary = stats_df.groupby('p_type').agg(
    Counts=('weight', 'count'),
    Weighted_Yield=('weight', 'sum')
).reset_index()

total_counts = summary['Counts'].sum()
total_weighted = summary['Weighted_Yield'].sum()

summary['Raw_%'] = (summary['Counts'] / total_counts) * 100
summary['Weighted_%'] = (summary['Weighted_Yield'] / total_weighted) * 100

# Sort by weighted yield (descending)
summary = summary.sort_values(by='Weighted_Yield', ascending=False)

# --- 3. Pretty Print Output ---
print("\n" + "="*60)
print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | {row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")

print("-" * 60)
print(f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | {total_weighted:<10.1f} | {100.0:<9.2f}%")
print("="*60 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Default bin definitions
DEFAULT_BINS_Y = np.linspace(0, 10, 51)  # dE/dx range [MeV/cm]
DEFAULT_BINS_Z = np.linspace(0, 200, 51) # Residual range [cm]


def plot_split_tpc_2d(
    df: pd.DataFrame,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    weight_col: str = None,
    bins_x: np.ndarray = DEFAULT_BINS_Z,
    bins_y: np.ndarray = DEFAULT_BINS_Y,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 2D histogram multiplot split by X < 0 (left) and X >= 0 (right)."""

    # Clean data & extract arrays safely
    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
    )
    x_vals, y_vals, split_vals, weights = (
        x_vals[finite_mask],
        y_vals[finite_mask],
        split_vals[finite_mask],
        weights[finite_mask],
    )

    # Subdivide by TPC side using x_split_col
    mask_neg_x = split_vals < 0
    mask_pos_x = split_vals >= 0

    # Setup 1x2 Subplots with shared Y-axis
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    cmap = globals().get("sunset_cmap", cmap_name)

    # Calculate global max for uniform colorbar scaling
    h_left, _, _ = np.histogram2d(
        x_vals[mask_neg_x], y_vals[mask_neg_x], bins=[bins_x, bins_y], weights=weights[mask_neg_x]
    )
    h_right, _, _ = np.histogram2d(
        x_vals[mask_pos_x], y_vals[mask_pos_x], bins=[bins_x, bins_y], weights=weights[mask_pos_x]
    )
    vmax = max(h_left.max(), h_right.max())
    vmax = vmax if vmax > 0 else None

    # --- Left Plot: Split Var < 0 ---
    im0 = ax_left.hist2d(
        x_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_left.set_title(f"{title_prefix}: $X < 0$ cm", fontsize=14, pad=10)
    ax_left.set_xlabel(xlabel, fontsize=14)
    ax_left.set_ylabel(ylabel, fontsize=14)
    ax_left.set_xlim(bins_x[0], bins_x[-1])
    ax_left.set_ylim(bins_y[0], bins_y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: Split Var >= 0 ---
    im1 = ax_right.hist2d(
        x_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_right.set_title(f"{title_prefix}: $X \\geq 0$ cm", fontsize=14, pad=10)
    ax_right.set_xlabel(xlabel, fontsize=14)
    ax_right.set_xlim(bins_x[0], bins_x[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Common Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label("Weighted Entries", fontsize=12)

    return fig, (ax_left, ax_right)

In [ ]:
# Pass plain string column names
for name, hitdf in hit_dfs.items():
    fig, axes = plot_split_tpc_2d(
        df=hitdf,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        weight_col=None,
        bins_x=np.linspace(0, 80, 41),   # Residual Range [cm]
        bins_y=np.linspace(0, 10, 41),    # dE/dx [MeV/cm]
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {name} dE/dx vs RR",
    )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    stacked: bool = False,
    density: bool = False,  # 👈 Added parameter
    alpha: float = 0.3,
    linewidth: float = 1.8,
    figsize: tuple = (8, 6),
    title: str = None,
    ax: plt.Axes = None,
):
    # --- 1. Filter RR Range & Clean Data ---
    mask = (
        df[dedx_col].notna()
        & df[rr_col].notna()
        & df[x_col].notna()
        & (df[rr_col] >= rr_range[0])
        & (df[rr_col] < rr_range[1])
    )
    plot_df = df[mask].copy()

    if plot_df.empty:
        raise ValueError(
            f"No valid entries found for {rr_col} in range {rr_range}."
        )

    # Resolve Weights
    if weight_col is not None and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=plot_df.index)

    # --- 2. Categorization by X Position ---
    mask_pos_x = plot_df[x_col] > 0
    mask_neg_x = ~mask_pos_x

    grouped_data = [
        plot_df.loc[mask_pos_x, dedx_col],
        plot_df.loc[mask_neg_x, dedx_col],
    ]
    grouped_weights = [
        weights[mask_pos_x],
        weights[mask_neg_x],
    ]

    # --- Handle Normalization (density=True) ---
    bin_width = bins[1] - bins[0]
    if density:
        if stacked:
            # Normalize so the combined stacked area sums to 1.0
            total_weight = sum(w.sum() for w in grouped_weights)
            if total_weight > 0:
                scale = 1.0 / (total_weight * bin_width)
                grouped_weights = [w * scale for w in grouped_weights]
        else:
            # Normalize each subgroup independently so each group's area sums to 1.0
            normed_weights = []
            for w in grouped_weights:
                w_sum = w.sum()
                if w_sum > 0:
                    normed_weights.append(w / (w_sum * bin_width))
                else:
                    normed_weights.append(w)
            grouped_weights = normed_weights

    colors = ["#1f77b4", "#d62728"]  # Blue & Red
    labels = [r"$X > 0$ cm", r"$X \leq 0$ cm"]

    # --- 3. Plotting ---
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    # Layer 1: Filled steps
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    # Layer 2: Outlines
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # --- 4. Styling & Formatting ---
    ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=14)

    if density:
        ax.set_ylabel("A.U.", fontsize=14)
    elif weight_col:
        ax.set_ylabel("Weighted Hits", fontsize=14)
    else:
        ax.set_ylabel("Hits", fontsize=14)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax.set_title(title, fontsize=15, pad=12)

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

    ax.tick_params(axis="both", which="both", labelsize=12, direction="in")
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    ax.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    return fig, ax

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(2, 3), (6, 7), (15, 16)]

for rr_min, rr_max in rr_ranges:
    fig, axes = plt.subplots(
        1, len(hit_dfs), figsize=(18, 5), sharey=True, gridspec_kw={"wspace": 0.08}
    )

    max_y_value = 0  # Track global maximum density across planes

    for plane_idx, (df_plane, ax) in enumerate(zip(hit_dfs, axes)):
        try:
            plot_dedx_by_rr_and_tpc(
                df=df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 51),
                stacked=False,
                density=True,  # 👈 Now supported!
                title=f"Plane {plane_idx}",
                ax=ax,
            )

            # Record maximum bin height in this subplot
            current_max = ax.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax.set_title(f"Plane {plane_idx}: No Data", fontsize=15, pad=12)
            continue

        # Clean up side subplots
        if plane_idx > 0:
            ax.set_ylabel("")
            legend = ax.get_legend()
            if legend:
                legend.remove()

    # Apply the global max limit + 15% headroom to all subplots
    if max_y_value > 0:
        axes[0].set_ylim(0, max_y_value * 1.15)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=1.03,
    )

    plt.show()

In [ ]:
hfit = load_physics_classes()

In [ ]:
pdg = 13
particle="muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"
    

all_results, fit_params = analyze_theoretical(hit_dfs,hfit, pdg,  particle=particle)

In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.signal import fftconvolve

def gaussian_kernel(x, sigma):
    """Zero-mean Gaussian sampled on x, normalized to unit area by
    the caller (not here, since spacing dx isn't known inside this fn)."""
    if sigma <= 0:
        kernel = np.zeros_like(x)
        kernel[np.argmin(np.abs(x))] = 1.0
        return kernel
    return np.exp(-0.5 * (x / sigma) ** 2)


def build_pdf_grid(pdf, xmin=-5.0, xmax=25.0, n_points=6001):
    """Evaluate the theoretical TF1 PDF once on a fixed fine grid.
    Reused across all sigma trials during the fit -- only the Gaussian
    kernel and the convolution are recomputed per curve_fit iteration."""
    x_grid = np.linspace(xmin, xmax, n_points)
    pdf_vals = np.array([pdf.Eval(x) for x in x_grid])
    dx = x_grid[1] - x_grid[0]
    return x_grid, pdf_vals, dx


def robust_max_x_py(tf1, xmin, xmax, n_points=2000):
    """Python port of the C++ robust_max_x: scan a grid rather than trust
    TF1::GetMaximumX(), which can get stuck depending on start point."""
    xs = np.linspace(xmin, xmax, n_points)
    vals = np.array([tf1.Eval(x) for x in xs])
    return xs[np.argmax(vals)]


def convolved_theoretical_pdf(x_query, sigma, amplitude, x_grid, pdf_vals, dx):
    """model(x) = amplitude * (theoretical PDF (x) Gaussian(0, sigma))(x).

    x_grid/pdf_vals/dx are the *fixed* physics PDF (pitch=0.32, rr=slice
    center) -- only sigma and amplitude change during the fit.
    """
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)  # == unit-area Gaussian
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return amplitude * np.interp(x_query, x_grid, conv)


def fit_theoretical_conv_slice(
    bin_centers,
    bin_counts,
    bin_errs,
    x_grid,
    pdf_vals,
    dx,
    mpv_theory,
    fit_range=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
):
    """curve_fit wrapper: only (sigma, amplitude) float. Fit range defaults
    to [0.7, 1.5] x mpv_theory, matching the Langau Stage-2 window."""
    if fit_range is None:
        fit_range = (0.7 * mpv_theory, 1.5 * mpv_theory)

    mask = (bin_centers >= fit_range[0]) & (bin_centers <= fit_range[1]) & (bin_counts > 0)
    if mask.sum() < 5:
        return None

    xs, ys, yerr = bin_centers[mask], bin_counts[mask], bin_errs[mask]
    yerr = np.where(yerr > 0, yerr, 1.0)

    # == Rough amplitude start: match data peak height to the (unsmeared)
    # == PDF peak height -- curve_fit refines both params from here.
    amp0 = ys.max() / max(pdf_vals.max(), 1e-9)

    def model(x, sigma, amplitude):
        return convolved_theoretical_pdf(x, sigma, amplitude, x_grid, pdf_vals, dx)

    try:
        popt, pcov = curve_fit(
            model, xs, ys, p0=[sigma0, amp0],
            sigma=yerr, absolute_sigma=True,
            bounds=([sigma_bounds[0], 0.0], [sigma_bounds[1], np.inf]),
            maxfev=20000,
        )
    except Exception:
        return None

    perr = np.sqrt(np.diag(pcov))
    sigma_fit, amp_fit = popt
    sigma_err, amp_err = perr
    return dict(
        sigma=sigma_fit, sigma_err=sigma_err,
        amplitude=amp_fit, amplitude_err=amp_err,
        fit_range=fit_range,
    )

def analyze_theoretical(
    hit_dfs,
    hfit,
    pdg,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    pitch=0.32,
    mass=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
    max_sigma_err=0.2,
    out_prefix="langau_rr_theoretical",
    make_summary_plots=True,
    verbose=True,
):
    """Analyzes a set of hit DataFrames across planes and TPCs, fitting the
    fixed theoretical PDF (x) zero-mean-Gaussian(sigma) to each rr slice
    instead of a free 4-parameter Langau.

    In addition to the per-TPC fits (tpc = 0, 1), each plane is also fit
    with both TPCs combined, stored under tpc = -1 -- same convention as
    `analyze`.
    """
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 40.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    fit_params = {}

    for plane, df in enumerate(hit_dfs):
        # tpc == -1 means "both TPCs combined": fit the full per-plane
        # dataframe instead of filtering down to a single TPC.
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[theoretical] Plane {plane}, TPC {tpc_label}")

            sub = df if tpc == -1 else df[df["tpc"] == tpc]

            res = fit_rr_slices_theoretical(
                sub,
                plane,
                tpc,
                hfit=hfit,
                pdg=pdg,
                dedx_col=dedx_col,
                rr_max=rr_max,
                pitch=pitch,
                mass=mass,
                sigma0=sigma0,
                sigma_bounds=sigma_bounds,
                max_sigma_err=max_sigma_err,
                verbose=verbose,
            )
            all_results[(plane, tpc)] = res

            popt = perr = None
            if len(res) >= 3:
                try:
                    popt, perr = fit_sigma_vs_mpv(
                        res, x_col="mpv_theory", y_col="gsigma", yerr_col="gsigma_err"
                    )
                except RuntimeError as exc:
                    if verbose:
                        print(f"  power-law fit failed: {exc}")
            fit_params[(plane, tpc)] = (popt, perr)

    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )

    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )

    return all_results, fit_params




In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PLANE_COLORS = ["tab:red", "tab:blue", "tab:green"]
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


def fit_rr_slices_theoretical(
    df,
    plane,
    tpc,
    hfit,
    pdg,
    dedx_col="dedx",
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    hist_nbins=150,
    hist_xmin=0.0,
    hist_xmax=20.0,
    min_entries=30,
    pitch=0.32,  # == fixed, per the hardcoded C++ pitch / distribution mode
    mass=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
    max_sigma_err=0.2,
    verbose=False,
):
    """Same rr-slicing loop as fit_rr_slices, but fits the fixed theoretical
    PDF (x) zero-mean-Gaussian(sigma) instead of a free 4-parameter Langau.

    Output columns are aligned with fit_rr_slices's output (mpv_x/mpv_x_err
    is the theoretical MPV here, mpv_x_err fixed at 0 since it isn't a
    fitted quantity) so downstream code (fit_sigma_vs_mpv, plot_all_planes,
    plot_by_plane) works unchanged on either analyze(...) or
    analyze_theoretical(...) output.
    """
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = df[(df["rr"] >= lo) & (df["rr"] < hi)]
        if len(sl) < min_entries:
            continue
        rr_center = 0.5 * (lo + hi)

        pdf = build_theoretical_pdf(hfit, pdg, rr_center, pitch, mass=mass)
        mpv_theory = robust_max_x_py(pdf, 0.0, 10.0, 2000)
        x_grid, pdf_vals, dx = build_pdf_grid(pdf)

        counts, bin_edges = np.histogram(
            sl[dedx_col].to_numpy(), bins=hist_nbins, range=(hist_xmin, hist_xmax)
        )
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_errs = np.where(counts > 0, np.sqrt(counts), 1.0)

        fit = fit_theoretical_conv_slice(
            bin_centers, counts.astype(float), bin_errs,
            x_grid, pdf_vals, dx, mpv_theory,
            sigma0=sigma0, sigma_bounds=sigma_bounds,
        )
        if fit is None:
            if verbose:
                print(f"   [plane {plane}, tpc {tpc}] rr={rr_center:.2f}: fit failed")
            continue
        if fit["sigma_err"] > max_sigma_err or not np.isfinite(fit["sigma_err"]):
            if verbose:
                print(f"   [plane {plane}, tpc {tpc}] rr={rr_center:.2f}: "
                      f"dropped, sigma_err={fit['sigma_err']:.3f}")
            continue

        rows.append(dict(
            plane=plane,
            tpc=tpc,
            rr_center=rr_center,
            n_hits=len(sl),
            # == mpv_x/mpv_x_err: same names fit_rr_slices uses, so
            # == downstream code (fit_sigma_vs_mpv, plotting) is identical
            # == regardless of which fitter produced the dataframe.
            mpv_x=mpv_theory,
            mpv_x_err=0.0,
            # == kept too, for anything that wants the theoretical value by
            # == its more explicit name
            mpv_theory=mpv_theory,
            width=np.nan,       # == no free Landau width in this fit
            width_err=np.nan,
            area=fit["amplitude"],
            area_err=fit["amplitude_err"],
            gsigma=fit["sigma"],
            gsigma_err=fit["sigma_err"],
            chi2=np.nan,        # == not computed by curve_fit here
            ndf=np.nan,
            status=np.nan,
            fit_range_lo=fit["fit_range"][0],
            fit_range_hi=fit["fit_range"][1],
        ))

    return pd.DataFrame(rows)


def analyze_theoretical(
    hit_dfs,
    hfit,
    pdg,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    pitch=0.32,
    mass=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
    max_sigma_err=0.2,
    out_prefix="langau_rr_theoretical",
    verbose=True,
):
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 40.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    sigma_fit_params = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[theoretical] Plane {plane}, TPC {tpc_label}")

            sub = df if tpc == -1 else df[df["tpc"] == tpc]

            res = fit_rr_slices_theoretical(
                sub,
                plane,
                tpc,
                hfit=hfit,
                pdg=pdg,
                dedx_col=dedx_col,
                rr_max=rr_max,
                pitch=pitch,
                mass=mass,
                sigma0=sigma0,
                sigma_bounds=sigma_bounds,
                max_sigma_err=max_sigma_err,
                verbose=verbose,
            )
            all_results[(plane, tpc)] = res

            popt_sig = perr_sig = None

            if len(res) >= 3:
                try:
                    popt_sig, perr_sig = fit_sigma_vs_mpv(
                        res,
                        x_col="mpv_theory",
                        y_col="gsigma",
                        yerr_col="gsigma_err",
                    )
                except Exception:
                    popt_sig, perr_sig = None, None

            sigma_fit_params[(plane, tpc)] = (popt_sig, perr_sig)

    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )

    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )

    return all_results, sigma_fit_params


def plot_all_planes(all_results, fit_params, particle, out_prefix):
    """Plots all (plane, tpc) power-law fit curves on a single square plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS[plane]
        style_info = TPC_STYLES.get(
            tpc,
            {"linestyle": "-", "label_prefix": f"tpc={tpc}"},
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["mpv_x"].min(), res["mpv_x"].max(), 200)
        label = f"{prefix}, Plane {plane}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
        ax.plot(
            xs,
            power_law(xs, *popt),
            linestyle=style,
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
    ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

    ax.legend(fontsize=10, loc="upper left", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)

    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


def plot_by_plane(
    all_results,
    fit_params,
    particle="muon",
    out_prefix="comparison",
    x_limits=None,
):
    """One square subplot per plane with TPC 0, 1, and Combined overlaid."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_visible_min, y_visible_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt, _ = fit_params.get((plane, tpc), (None, None))

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]
            style = style_info["linestyle"]
            marker = style_info["marker"]
            prefix = style_info["label_prefix"]

            if x_limits is not None:
                mask = (res["mpv_x"] >= x_limits[0]) & (
                    res["mpv_x"] <= x_limits[1]
                )
                res_filtered = res[mask]
            else:
                res_filtered = res

            if len(res_filtered):
                ax.errorbar(
                    res_filtered["mpv_x"],
                    res_filtered["gsigma"],
                    yerr=res_filtered["gsigma_err"],
                    fmt=marker,
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{prefix} Data",
                )
                if x_limits is not None:
                    y_visible_min = min(
                        y_visible_min,
                        (
                            res_filtered["gsigma"] - res_filtered["gsigma_err"]
                        ).min(),
                    )
                    y_visible_max = max(
                        y_visible_max,
                        (
                            res_filtered["gsigma"] + res_filtered["gsigma_err"]
                        ).max(),
                    )

            if popt is not None:
                x_min = x_limits[0] if x_limits else res["mpv_x"].min()
                x_max = x_limits[1] if x_limits else res["mpv_x"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = power_law(xs, *popt)

                label_fit = f"{prefix}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
                ax.plot(
                    xs,
                    ys,
                    linestyle=style,
                    color=color,
                    linewidth=1.8,
                    label=label_fit,
                )

                if x_limits is not None:
                    y_visible_min = min(y_visible_min, ys.min())
                    y_visible_max = max(y_visible_max, ys.max())

        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_visible_min) and not np.isinf(y_visible_max):
                y_pad = (y_visible_max - y_visible_min) * 0.1
                ax.set_ylim(y_visible_min - y_pad, y_visible_max + y_pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
        ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

        ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)

        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_by_plane{zoom_suffix}.pdf")
    plt.show()

In [ ]:
sigma_fit_params

In [ ]:
# 1. Run analysis (now returns 2 objects)
all_results, sigma_fit_params = analyze_theoretical(
    hit_dfs,
    hfit,
    pdg,
    particle=particle,
)

# 2. Plot sigma across all planes
plot_all_planes(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison",
)

# 3. Plot sigma by plane (full range)
plot_by_plane(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison",
)

# 4. Plot sigma by plane (zoomed range)
plot_by_plane(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison",
    x_limits=[1.9, 3.0],
)

In [ ]:
import numpy as np
from scipy.signal import savgol_filter
'''
def robust_hist_max(
    values, 
    xmin, 
    xmax, 
    nbins=500, 
    savgol_window=51, 
    savgol_poly=3, 
    n_boot=200
):
    """
    Finds the mode of a dataset using Savitzky-Golay filtering, sub-bin 
    parabolic interpolation, and Poisson bootstrap uncertainty.

    Returns:
        x_max     : Refined peak position
        x_max_err : Standard uncertainty on peak position (0.0 if n_boot=0)
        centers   : Histogram bin center values
        counts    : Raw histogram counts
        smoothed  : Savitzky-Golay smoothed counts
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")

    counts, edges = np.histogram(vals, bins=nbins, range=(xmin, xmax))
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = centers[1] - centers[0]

    def _extract_peak(cnts):
        n_bins = len(cnts)
        
        # Fall back if array is too small for polyorder
        if n_bins <= savgol_poly + 1:
            i_max = int(np.argmax(cnts))
            return centers[i_max], cnts.astype(float)

        w = min(savgol_window, n_bins)
        if w % 2 == 0:
            w -= 1
        if w <= savgol_poly:
            w = savgol_poly + 2 if (savgol_poly % 2 == 1) else savgol_poly + 1

        if w > n_bins or w <= savgol_poly:
            i_max = int(np.argmax(cnts))
            return centers[i_max], cnts.astype(float)

        smoothed_arr = savgol_filter(
            cnts.astype(float), 
            window_length=w, 
            polyorder=savgol_poly, 
            mode='nearest'
        )
        i_max = int(np.argmax(smoothed_arr))

        # 3-point parabolic refinement
        if 0 < i_max < n_bins - 1:
            y0, y1, y2 = smoothed_arr[i_max - 1], smoothed_arr[i_max], smoothed_arr[i_max + 1]
            denom = y0 - 2.0 * y1 + y2
            
            # Require concave downward peak (denom < 0)
            if denom < 0:
                delta = 0.5 * (y0 - y2) / denom
                delta = np.clip(delta, -0.5, 0.5)
                x_peak = centers[i_max] + delta * bin_width
            else:
                x_peak = centers[i_max]
        else:
            x_peak = centers[i_max]

        return x_peak, smoothed_arr

    # Primary estimate
    x_max, smoothed = _extract_peak(counts)

    # Uncertainty estimation via Poisson Monte Carlo
    x_max_err = 0.0
    if n_boot > 0 and np.sum(counts) > 0:
        poisson_counts = np.random.poisson(counts, size=(n_boot, len(counts)))
        boot_peaks = np.empty(n_boot)
        for b in range(n_boot):
            boot_peaks[b], _ = _extract_peak(poisson_counts[b])
        x_max_err = float(np.std(boot_peaks))

    return x_max, x_max_err, centers, counts, smoothed


import numpy as np
from scipy.stats import gaussian_kde


def kde_hist_max(
    values,
    xmin,
    xmax,
    bw_method=None,
    grid_points=2000,
    n_boot=200,
):
    """
    Finds the mode of a dataset using a Gaussian KDE instead of a fine
    histogram + Savitzky-Golay smoothing.

    KDE avoids the "counts per bin" noise problem entirely: instead of
    binning into (e.g.) 1000 narrow bins with ~20 events each and then
    smoothing, every event contributes a smooth little bump, and the
    sum is smooth everywhere. That tends to be much more stable for
    Landau-like peaks with a sharp rise and long tail, where fine-bin
    Poisson noise near the crest can pull a 3-point parabola off-center.

    bw_method: None uses Scott's rule (scipy default). If the KDE peak
    looks too smoothed-out or too spiky, pass a float like 0.05-0.15
    to control bandwidth directly (in units of xmax-xmin, roughly).

    Returns: x_max, x_max_err, grid (x), density (y), density (again,
    kept as 5th slot so it's a drop-in replacement for robust_hist_max's
    (x_max, x_max_err, centers, counts, smoothed) signature).
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = vals[(vals >= xmin) & (vals <= xmax)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")

    def _peak_from_sample(sample):
        kde = gaussian_kde(sample, bw_method=bw_method)
        grid = np.linspace(xmin, xmax, grid_points)
        density = kde(grid)
        i_max = int(np.argmax(density))
        dx = grid[1] - grid[0]
        if 0 < i_max < grid_points - 1:
            y0, y1, y2 = density[i_max - 1], density[i_max], density[i_max + 1]
            denom = y0 - 2.0 * y1 + y2
            if denom < 0:
                delta = 0.5 * (y0 - y2) / denom
                delta = np.clip(delta, -0.5, 0.5)
                x_peak = grid[i_max] + delta * dx
            else:
                x_peak = grid[i_max]
        else:
            x_peak = grid[i_max]
        return x_peak, grid, density

    x_max, grid, density = _peak_from_sample(vals)

    x_max_err = 0.0
    if n_boot > 0 and len(vals) > 1:
        boot_peaks = np.empty(n_boot)
        n = len(vals)
        for b in range(n_boot):
            sample = vals[np.random.randint(0, n, size=n)]
            boot_peaks[b], _, _ = _peak_from_sample(sample)
        x_max_err = float(np.std(boot_peaks))

    return x_max, x_max_err, grid, density, density


def local_quad_hist_max(
    values,
    xmin,
    xmax,
    nbins=150,
    half_window_bins=6,
    n_boot=200,
):
    """
    Finds the mode via a coarser histogram (more counts/bin, less noise)
    plus a least-squares quadratic fit over a window of bins around the
    raw argmax, rather than a 3-point parabola on a savgol-smoothed,
    very finely binned histogram.

    Using np.polyfit over ~2*half_window_bins+1 points is more robust to
    single noisy bins than the 3-point formula, since it's a least-squares
    fit rather than an exact interpolation through 3 points.

    Returns: x_max, x_max_err, centers, counts, fitted-curve-over-window
    (last slot is the quadratic evaluated on the window, for plotting/
    diagnostics -- not the same length as counts, so treat as a
    (x_window, y_window_fit) tuple).
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")

    def _peak_from_sample(sample):
        counts, edges = np.histogram(sample, bins=nbins, range=(xmin, xmax))
        centers = 0.5 * (edges[:-1] + edges[1:])
        i_max = int(np.argmax(counts))
        lo = max(0, i_max - half_window_bins)
        hi = min(len(counts), i_max + half_window_bins + 1)
        x_win = centers[lo:hi]
        y_win = counts[lo:hi].astype(float)
        if len(x_win) < 3:
            return centers[i_max], centers, counts, (x_win, y_win)
        a, b, c = np.polyfit(x_win, y_win, 2)
        if a < 0:
            x_peak = -b / (2.0 * a)
            # guard against a wild extrapolated vertex outside the window
            if not (x_win[0] <= x_peak <= x_win[-1]):
                x_peak = centers[i_max]
        else:
            x_peak = centers[i_max]
        return x_peak, centers, counts, (x_win, y_win)

    x_max, centers, counts, window_fit = _peak_from_sample(vals)

    x_max_err = 0.0
    if n_boot > 0 and len(vals) > 1:
        boot_peaks = np.empty(n_boot)
        n = len(vals)
        for b in range(n_boot):
            sample = vals[np.random.randint(0, n, size=n)]
            boot_peaks[b], _, _, _ = _peak_from_sample(sample)
        x_max_err = float(np.std(boot_peaks))

    return x_max, x_max_err, centers, counts, window_fit
'''

In [ ]:
def convolve_pdf_with_gaussian(
    pdf, sigma_g, x_eval, n_sigma=5.0, grid_step=None, recenter_to_mode=False
):
    """Numerically convolves `pdf` with a Gaussian of width `sigma_g`.

    If recenter_to_mode=True, the convolved curve is shifted along x so its
    peak lines up with the peak of the unconvolved `pdf`. This is a purely
    cosmetic re-alignment for shape comparisons -- physically, convolving a
    skewed (Landau) PDF with a Gaussian *does* shift the mode, and that
    shift is real detector-resolution physics, not a bug in the convolution.
    """
    if sigma_g is None or not np.isfinite(sigma_g) or sigma_g <= 0:
        return np.array([pdf.Eval(x) for x in x_eval])

    x_eval = np.asarray(x_eval, dtype=float)

    if grid_step is None:
        grid_step = sigma_g / 25.0

    n_pad = int(np.ceil((n_sigma * sigma_g) / grid_step))
    pad_width = n_pad * grid_step

    grid_min = x_eval.min() - pad_width
    grid_max = x_eval.max() + pad_width
    grid = np.arange(grid_min, grid_max + grid_step, grid_step)

    pdf_vals = np.array([pdf.Eval(t) for t in grid])

    k = np.arange(-n_pad, n_pad + 1)
    kernel_x = k * grid_step
    kernel = np.exp(-0.5 * (kernel_x / sigma_g) ** 2)
    kernel /= kernel.sum()

    conv_vals = np.convolve(pdf_vals, kernel, mode="same")

    if recenter_to_mode:
        # Measure how far the convolution moved the peak, then shift the
        # x-axis by that amount before sampling back onto x_eval.
        mode_pdf = grid[np.argmax(pdf_vals)]
        mode_conv = grid[np.argmax(conv_vals)]
        shift = mode_conv - mode_pdf
        return np.interp(x_eval + shift, grid, conv_vals)

    return np.interp(x_eval, grid, conv_vals)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

'''
def plot_slice_diagnostic(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    sigma_fit_params=None,
    shift_fit_params=None,  # Kept for interface compatibility
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    x_limits=None,
    first_stage_range=(0.0, 10.0),
    savgol_window=51,
    savgol_poly=3,
    peak_nbins=500,
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice.

    Plots data histogram and theoretical curves with vertical peak lines for
    each stage, featuring peak-locking directly to the data histogram peak
    extracted via Savitzky-Golay smoothing and parabolic interpolation
    (robust_hist_max).
    """
    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]
    rr_center = 0.5 * (rr_min + rr_max)

    fig, ax = plt.subplots(figsize=(9, 6))

    # --- Stage 1: Data Histogram ---
    data_vals = sl[dedx_col].dropna().to_numpy()
    counts, edges, _ = ax.hist(
        data_vals,
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"Data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 500)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])

    data_max = counts.max() if len(counts) > 0 else 0.0
    curve_max = data_max

    # 1. Data Peak Line using robust_hist_max
    data_peak_x = None
    if len(data_vals) > 0 and data_max > 0:
        try:
            data_peak_x, _, _, _ = robust_hist_max(
                data_vals,
                hist_xmin,
                hist_xmax,
                nbins=peak_nbins,
                savgol_window=savgol_window,
                savgol_poly=savgol_poly,
            )
        except Exception:
            data_peak_x = bin_centers[np.argmax(counts)]

        ax.axvline(
            data_peak_x,
            color="navy",
            linestyle=":",
            linewidth=1.8,
            label=f"Data Peak ({data_peak_x:.2f})",
        )

    if hfit is not None:
        mean_pitch = (
            sl["pitch"].median()
            if ("pitch" in sl and len(sl) > 0)
            else 0.32
        )
        pdf = build_theoretical_pdf(
            hfit, pdg, rr_center, mean_pitch, mass=mass
        )

        # --- Stage 2: Theory PDF (Unconvolved) ---
        theo_mpv = pdf.GetMaximumX()
        y_theo = np.array([pdf.Eval(xv) for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = (y_theo / y_theo.max()) * data_max

        ax.plot(
            x_eval,
            y_theo,
            color="red",
            linestyle=":",
            linewidth=2.5,
            label="1. Theory Unconvolved",
        )
        # 2. Theory MPV Peak Line
        ax.axvline(
            theo_mpv,
            color="red",
            linestyle="--",
            linewidth=1.5,
            alpha=0.8,
            label=f"Theory MPV ({theo_mpv:.2f})",
        )
        curve_max = max(curve_max, y_theo.max())

        # Retrieve resolution parameters
        popt_sig = (
            sigma_fit_params.get((plane, tpc), (None, None))[0]
            if sigma_fit_params
            else None
        )

        # --- Stage 3: Convolved PDF (Unshifted) ---
        if popt_sig is not None:
            predicted_sigma_g = power_law(theo_mpv, *popt_sig)
            y_pred_unsh = convolve_pdf_with_gaussian(
                pdf, predicted_sigma_g, x_eval, recenter_to_mode=False
            )
            if y_pred_unsh.max() > 0 and data_max > 0:
                y_pred_unsh = (y_pred_unsh / y_pred_unsh.max()) * data_max

            unsh_peak_x = x_eval[np.argmax(y_pred_unsh)]

            ax.plot(
                x_eval,
                y_pred_unsh,
                color="mediumpurple",
                linestyle="--",
                linewidth=2.0,
                label=r"2. Theory $\otimes$ Res (Unshifted)",
            )
            # 3. Unshifted Convolved Peak Line
            ax.axvline(
                unsh_peak_x,
                color="mediumpurple",
                linestyle="-.",
                linewidth=1.5,
                alpha=0.8,
                label=f"Unshifted Peak ({unsh_peak_x:.2f})",
            )
            curve_max = max(curve_max, y_pred_unsh.max())

            # --- Stage 4: Convolved PDF + Peak Aligned to Data ---
            if data_peak_x is not None:
                # Lock peak position directly to data histogram MPV derived from robust_hist_max
                shift_to_data = data_peak_x - unsh_peak_x
                y_pred_shifted = np.interp(
                    x_eval - shift_to_data, x_eval, y_pred_unsh
                )

                shf_peak_x = x_eval[np.argmax(y_pred_shifted)]

                ax.plot(
                    x_eval,
                    y_pred_shifted,
                    color="darkorange",
                    linestyle="-",
                    linewidth=2.5,
                    label=r"3. Theory $\otimes$ Res (Peak Locked)",
                )
                # 4. Shifted Peak Line
                ax.axvline(
                    shf_peak_x,
                    color="darkorange",
                    linestyle="-",
                    linewidth=1.5,
                    alpha=0.8,
                    label=f"Locked Peak ({shf_peak_x:.2f})",
                )
                curve_max = max(curve_max, y_pred_shifted.max())

    # --- Zoom Limits & Dynamic Y-Rescaling ---
    if x_limits is not None:
        ax.set_xlim(x_limits)
        mask = (x_eval >= x_limits[0]) & (x_eval <= x_limits[1])

        visible_max = data_max
        if "y_theo" in locals() and len(y_theo[mask]):
            visible_max = max(visible_max, y_theo[mask].max())
        if "y_pred_unsh" in locals() and len(y_pred_unsh[mask]):
            visible_max = max(visible_max, y_pred_unsh[mask].max())
        if "y_pred_shifted" in locals() and len(y_pred_shifted[mask]):
            visible_max = max(visible_max, y_pred_shifted[mask].max())

        ax.set_ylim(0, visible_max * 1.15 if visible_max > 0 else 1.0)
    else:
        ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]", fontsize=12)
    ax.set_ylabel(f"Hits / {bin_width:.2f} MeV/cm", fontsize=12)

    tpc_title = "TPCs Combined" if tpc == -1 else f"TPC {tpc}"
    ax.set_title(
        f"Plane {plane}, {tpc_title}, {rr_min:g} <= RR < {rr_max:g} cm",
        fontsize=13,
    )

    ax.legend(fontsize=8, loc="upper right", framealpha=0.9, ncol=2)
    ax.grid(True, linestyle=":", alpha=0.5)
    fig.tight_layout()

    return fig
'''



In [ ]:
import numpy as np
from scipy.signal import savgol_filter
from scipy.stats import gaussian_kde


# ---------------------------------------------------------------------------
# Method 1: your original Savitzky-Golay + 3-point parabola approach
# ---------------------------------------------------------------------------
def robust_hist_max(
    values,
    xmin,
    xmax,
    nbins=1000,
    savgol_window=51,
    savgol_poly=3,
    n_boot=200,
):
    """
    Finds the mode of a dataset using Savitzky-Golay filtering, sub-bin
    parabolic interpolation, and Poisson bootstrap uncertainty.
    Returns:
        x_max     : Refined peak position
        x_max_err : Standard uncertainty on peak position (0.0 if n_boot=0)
        centers   : Histogram bin center values
        counts    : Raw histogram counts
        smoothed  : Savitzky-Golay smoothed counts
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")
    counts, edges = np.histogram(vals, bins=nbins, range=(xmin, xmax))
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = centers[1] - centers[0]

    def _extract_peak(cnts):
        n_bins = len(cnts)
        if n_bins <= savgol_poly + 1:
            i_max = int(np.argmax(cnts))
            return centers[i_max], cnts.astype(float)
        w = min(savgol_window, n_bins)
        if w % 2 == 0:
            w -= 1
        if w <= savgol_poly:
            w = savgol_poly + 2 if (savgol_poly % 2 == 1) else savgol_poly + 1
        if w > n_bins or w <= savgol_poly:
            i_max = int(np.argmax(cnts))
            return centers[i_max], cnts.astype(float)
        smoothed_arr = savgol_filter(
            cnts.astype(float), window_length=w, polyorder=savgol_poly, mode='nearest'
        )
        i_max = int(np.argmax(smoothed_arr))
        if 0 < i_max < n_bins - 1:
            y0, y1, y2 = smoothed_arr[i_max - 1], smoothed_arr[i_max], smoothed_arr[i_max + 1]
            denom = y0 - 2.0 * y1 + y2
            if denom < 0:
                delta = 0.5 * (y0 - y2) / denom
                delta = np.clip(delta, -0.5, 0.5)
                x_peak = centers[i_max] + delta * bin_width
            else:
                x_peak = centers[i_max]
        else:
            x_peak = centers[i_max]
        return x_peak, smoothed_arr

    x_max, smoothed = _extract_peak(counts)
    x_max_err = 0.0
    if n_boot > 0 and np.sum(counts) > 0:
        poisson_counts = np.random.poisson(counts, size=(n_boot, len(counts)))
        boot_peaks = np.empty(n_boot)
        for b in range(n_boot):
            boot_peaks[b], _ = _extract_peak(poisson_counts[b])
        x_max_err = float(np.std(boot_peaks))
    return x_max, x_max_err, centers, counts, smoothed


# ---------------------------------------------------------------------------
# Method 2: Gaussian KDE (coarse-then-refine grid, subsampled bootstrap)
# ---------------------------------------------------------------------------
import numpy as np
from scipy.signal import fftconvolve
import numpy as np
from scipy.signal import fftconvolve


def kde_hist_max(
    values,
    xmin,
    xmax,
    nbins=2000,
    bw=None,
    n_boot=200,
    **_ignored,
):
    """
    Fast KDE-equivalent mode finder via binning + FFT convolution with a
    Gaussian kernel, instead of scipy.stats.gaussian_kde's exact O(N*M)
    evaluation (sum a kernel over every data point, at every grid point).

    Cost breakdown here:
      - np.histogram: O(N), same as any histogram -- cheap regardless of N.
      - FFT convolution: O(nbins * log(nbins)), independent of N.
      - Bootstrap: Poisson-resamples the *histogram counts* (like your
        original robust_hist_max), so each of the n_boot iterations is just
        one more FFT convolution -- no re-fitting a KDE on raw data, no
        rebuilding kernel objects.

    bw: kernel bandwidth in data units (e.g. MeV/cm). If None, uses
    Silverman's rule of thumb on the raw values. If your peak still looks
    over/under-smoothed, pass bw explicitly -- start around 0.05-0.15 for
    a dE/dx-like range of 0-10.

    Returns the same 5-slot signature as robust_hist_max:
        x_max, x_max_err, centers, counts, smoothed
    so it's a drop-in replacement in _PEAK_FINDERS / plot_slice_diagnostic /
    compute_shift_diagnostics.
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    vals = vals[(vals >= xmin) & (vals <= xmax)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")

    counts, edges = np.histogram(vals, bins=nbins, range=(xmin, xmax))
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_width = centers[1] - centers[0]

    if bw is None:
        # Robust Silverman's rule: use A = min(std, IQR/1.34) rather than
        # bare std. Plain std is very sensitive to a long right-hand tail
        # (e.g. Landau-tailed dE/dx at high rr) -- a handful of far-tail
        # hits can inflate std enough to badly oversmooth the tight core
        # peak, which for an asymmetric distribution biases the estimated
        # maximum toward the heavier tail side. IQR is far less sensitive
        # to those outliers.
        std = np.std(vals, ddof=1)
        q75, q25 = np.percentile(vals, [75, 25])
        iqr = q75 - q25
        if iqr > 0:
            scale = min(std, iqr / 1.34)
        else:
            scale = std
        bw = 0.9 * scale * len(vals) ** (-1.0 / 5.0)
        bw = max(bw, bin_width)  # never smooth finer than a bin

    sigma_bins = bw / bin_width
    kernel_radius = max(1, int(np.ceil(4.0 * sigma_bins)))
    kx = np.arange(-kernel_radius, kernel_radius + 1)
    kernel = np.exp(-0.5 * (kx / sigma_bins) ** 2)
    kernel /= kernel.sum()

    def _peak(cnts):
        smoothed_arr = fftconvolve(cnts.astype(float), kernel, mode="same")
        i_max = int(np.argmax(smoothed_arr))
        if 0 < i_max < len(smoothed_arr) - 1:
            y0, y1, y2 = smoothed_arr[i_max - 1], smoothed_arr[i_max], smoothed_arr[i_max + 1]
            denom = y0 - 2.0 * y1 + y2
            if denom < 0:
                delta = np.clip(0.5 * (y0 - y2) / denom, -0.5, 0.5)
                return centers[i_max] + delta * bin_width, smoothed_arr
        return centers[i_max], smoothed_arr

    x_max, smoothed = _peak(counts)

    x_max_err = 0.0
    if n_boot > 0 and counts.sum() > 0:
        poisson_counts = np.random.poisson(counts, size=(n_boot, len(counts)))
        boot_peaks = np.empty(n_boot)
        for b in range(n_boot):
            boot_peaks[b], _ = _peak(poisson_counts[b])
        x_max_err = float(np.std(boot_peaks))

    return x_max, x_max_err, centers, counts, smoothed
# ---------------------------------------------------------------------------
# Method 3: coarser histogram + windowed least-squares quadratic fit
# ---------------------------------------------------------------------------
def local_quad_hist_max(
    values,
    xmin,
    xmax,
    nbins=150,
    half_window_bins=6,
    n_boot=200,
    **_ignored,
):
    """
    Coarser histogram (more counts/bin, less noise) plus a least-squares
    quadratic fit over a window of bins around the raw argmax, instead of
    an exact 3-point parabola on a savgol-smoothed, very finely binned
    histogram. More robust to a single noisy bin skewing the result.

    Returns the same 5-slot signature as robust_hist_max, with the last
    slot holding (x_window, y_window) instead of a smoothed curve.
    """
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        raise ValueError("Input values array contains no finite data.")

    def _peak_from_sample(sample):
        counts, edges = np.histogram(sample, bins=nbins, range=(xmin, xmax))
        centers = 0.5 * (edges[:-1] + edges[1:])
        i_max = int(np.argmax(counts))
        lo = max(0, i_max - half_window_bins)
        hi = min(len(counts), i_max + half_window_bins + 1)
        x_win = centers[lo:hi]
        y_win = counts[lo:hi].astype(float)
        if len(x_win) < 3:
            return centers[i_max], centers, counts, (x_win, y_win)
        a, b, c = np.polyfit(x_win, y_win, 2)
        if a < 0:
            x_peak = -b / (2.0 * a)
            if not (x_win[0] <= x_peak <= x_win[-1]):
                x_peak = centers[i_max]
        else:
            x_peak = centers[i_max]
        return x_peak, centers, counts, (x_win, y_win)

    x_max, centers, counts, window_fit = _peak_from_sample(vals)

    x_max_err = 0.0
    if n_boot > 0 and len(vals) > 1:
        boot_peaks = np.empty(n_boot)
        n = len(vals)
        for b in range(n_boot):
            sample = vals[np.random.randint(0, n, size=n)]
            boot_peaks[b], _, _, _ = _peak_from_sample(sample)
        x_max_err = float(np.std(boot_peaks))

    return x_max, x_max_err, centers, counts, window_fit


# Dispatch table for the three base methods. "hybrid" is handled separately
# in plot_slice_diagnostic since it needs to know rr_center to pick a branch.
_PEAK_FINDERS = {
    "savgol": robust_hist_max,
    "kde": kde_hist_max,
    "local_quad": local_quad_hist_max,
}


# ---------------------------------------------------------------------------
# Modified diagnostic plot: pass peak_method="savgol" | "kde" | "local_quad" | "hybrid"
# ---------------------------------------------------------------------------
def plot_slice_diagnostic(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    sigma_fit_params=None,
    shift_fit_params=None,  # Kept for interface compatibility
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    x_limits=None,
    first_stage_range=(0.0, 10.0),
    savgol_window=51,
    savgol_poly=3,
    peak_nbins=500,
    peak_method="hybrid",  # "savgol" | "kde" | "local_quad" | "hybrid"
    peak_method_kwargs=None,  # extra kwargs forwarded to the chosen finder
    hybrid_rr_threshold=9.0,  # rr_center above this -> kde, below/equal -> local_quad
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice.

    Plots data histogram and theoretical curves with vertical peak lines for
    each stage. The data peak is extracted via one of four interchangeable
    methods, selected with `peak_method`:
        "savgol"     -> robust_hist_max (Savitzky-Golay + 3-pt parabola)
        "kde"        -> kde_hist_max (Gaussian KDE)
        "local_quad" -> local_quad_hist_max (coarse hist + windowed quad fit)
        "hybrid"     -> local_quad_hist_max for rr_center <= hybrid_rr_threshold,
                        kde_hist_max for rr_center > hybrid_rr_threshold
                        (matches the empirical regions where each behaves best)

    `peak_method_kwargs` lets you pass method-specific overrides, e.g.
    peak_method="kde", peak_method_kwargs={"bw_method": 0.1}. For "hybrid",
    kwargs are forwarded to whichever branch actually runs for this slice.
    """
    valid_methods = set(_PEAK_FINDERS) | {"hybrid"}
    if peak_method not in valid_methods:
        raise ValueError(
            f"Unknown peak_method {peak_method!r}; choose from {sorted(valid_methods)}"
        )
    extra_kwargs = dict(peak_method_kwargs or {})

    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]
    rr_center = 0.5 * (rr_min + rr_max)

    # Resolve "hybrid" to a concrete method + finder for THIS slice, based on rr_center
    if peak_method == "hybrid":
        active_method = "kde" if rr_center > hybrid_rr_threshold else "local_quad"
    else:
        active_method = peak_method
    finder = _PEAK_FINDERS[active_method]

    fig, ax = plt.subplots(figsize=(9, 6))

    # --- Stage 1: Data Histogram ---
    data_vals = sl[dedx_col].dropna().to_numpy()
    counts, edges, _ = ax.hist(
        data_vals,
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"Data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 500)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])

    data_max = counts.max() if len(counts) > 0 else 0.0
    curve_max = data_max

    # 1. Data Peak Line, computed via the resolved method for this slice
    data_peak_x = None
    if len(data_vals) > 0 and data_max > 0:
        try:
            call_kwargs = dict(extra_kwargs)
            if active_method == "savgol":
                call_kwargs.setdefault("nbins", peak_nbins)
                call_kwargs.setdefault("savgol_window", savgol_window)
                call_kwargs.setdefault("savgol_poly", savgol_poly)
            # kde / local_quad take their own defaults unless overridden
            # via peak_method_kwargs

            data_peak_x, _, _, _, _ = finder(
                data_vals, hist_xmin, hist_xmax, **call_kwargs
            )
        except Exception:
            data_peak_x = bin_centers[np.argmax(counts)]

        label_tag = (
            f"hybrid\u2192{active_method}" if peak_method == "hybrid" else active_method
        )
        ax.axvline(
            data_peak_x,
            color="navy",
            linestyle=":",
            linewidth=1.8,
            label=f"Data Peak [{label_tag}] ({data_peak_x:.2f})",
        )

    if hfit is not None:
        mean_pitch = 0.32
        pdf = build_theoretical_pdf(
            hfit, pdg, rr_center, mean_pitch, mass=mass
        )

        # --- Stage 2: Theory PDF (Unconvolved) ---
        theo_mpv = pdf.GetMaximumX()
        y_theo = np.array([pdf.Eval(xv) for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = (y_theo / y_theo.max()) * data_max

        ax.plot(
            x_eval,
            y_theo,
            color="red",
            linestyle=":",
            linewidth=2.5,
            label="1. Theory Unconvolved",
        )
        # 2. Theory MPV Peak Line
        ax.axvline(
            theo_mpv,
            color="red",
            linestyle="--",
            linewidth=1.5,
            alpha=0.8,
            label=f"Theory MPV ({theo_mpv:.2f})",
        )
        curve_max = max(curve_max, y_theo.max())

        # Retrieve resolution parameters
        popt_sig = (
            sigma_fit_params.get((plane, tpc), (None, None))[0]
            if sigma_fit_params
            else None
        )

        # --- Stage 3: Convolved PDF (Unshifted) ---
        if popt_sig is not None:
            predicted_sigma_g = power_law(theo_mpv, *popt_sig)
            y_pred_unsh = convolve_pdf_with_gaussian(
                pdf, predicted_sigma_g, x_eval, recenter_to_mode=False
            )
            if y_pred_unsh.max() > 0 and data_max > 0:
                y_pred_unsh = (y_pred_unsh / y_pred_unsh.max()) * data_max

            unsh_peak_x = x_eval[np.argmax(y_pred_unsh)]

            ax.plot(
                x_eval,
                y_pred_unsh,
                color="mediumpurple",
                linestyle="--",
                linewidth=2.0,
                label=r"2. Theory $\otimes$ Res (Unshifted)",
            )
            # 3. Unshifted Convolved Peak Line
            ax.axvline(
                unsh_peak_x,
                color="mediumpurple",
                linestyle="-.",
                linewidth=1.5,
                alpha=0.8,
                label=f"Unshifted Peak ({unsh_peak_x:.2f})",
            )
            curve_max = max(curve_max, y_pred_unsh.max())

            # --- Stage 4: Convolved PDF + Peak Aligned to Data ---
            if data_peak_x is not None:
                shift_to_data = data_peak_x - unsh_peak_x
                y_pred_shifted = np.interp(
                    x_eval - shift_to_data, x_eval, y_pred_unsh
                )

                # shift_to_data was calibrated so the shifted curve's peak
                # sits exactly at data_peak_x -- re-deriving it via argmax
                # on x_eval just reintroduces grid-quantization error
                # (worse the coarser x_eval is, e.g. when hist_xmax is large
                # relative to the fixed 500-point grid). Use it directly.
                shf_peak_x = data_peak_x

                ax.plot(
                    x_eval,
                    y_pred_shifted,
                    color="darkorange",
                    linestyle="-",
                    linewidth=2.5,
                    label=r"3. Theory $\otimes$ Res (Peak Locked)",
                )
                # 4. Shifted Peak Line
                ax.axvline(
                    shf_peak_x,
                    color="darkorange",
                    linestyle="-",
                    linewidth=1.5,
                    alpha=0.8,
                    label=f"Locked Peak ({shf_peak_x:.2f})",
                )
                curve_max = max(curve_max, y_pred_shifted.max())

    # --- Zoom Limits & Dynamic Y-Rescaling ---
    if x_limits is not None:
        ax.set_xlim(x_limits)
        mask = (x_eval >= x_limits[0]) & (x_eval <= x_limits[1])

        visible_max = data_max
        if "y_theo" in locals() and len(y_theo[mask]):
            visible_max = max(visible_max, y_theo[mask].max())
        if "y_pred_unsh" in locals() and len(y_pred_unsh[mask]):
            visible_max = max(visible_max, y_pred_unsh[mask].max())
        if "y_pred_shifted" in locals() and len(y_pred_shifted[mask]):
            visible_max = max(visible_max, y_pred_shifted[mask].max())

        ax.set_ylim(0, visible_max * 1.15 if visible_max > 0 else 1.0)
    else:
        ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]", fontsize=12)
    ax.set_ylabel(f"Hits / {bin_width:.2f} MeV/cm", fontsize=12)

    tpc_title = "TPCs Combined" if tpc == -1 else f"TPC {tpc}"
    method_tag = f"hybrid\u2192{active_method}" if peak_method == "hybrid" else peak_method
    ax.set_title(
        f"Plane {plane}, {tpc_title}, {rr_min:g} <= RR < {rr_max:g} cm "
        f"[{method_tag}]",
        fontsize=13,
    )

    ax.legend(fontsize=8, loc="upper right", framealpha=0.9, ncol=2)
    ax.grid(True, linestyle=":", alpha=0.5)
    fig.tight_layout()

    return fig

In [ ]:
import os
#"savgol" | "kde" | "local_quad"

output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)

hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(1, 2), (2, 3), (3, 4), (4, 5), (6, 7), (7, 8), (8, 9), (9,10), (6, 7), (15, 16), (30, 31)]

plane = 2
df = hit_dfs[plane]

for rr_min, rr_max in rr_ranges:
    fig1 = plot_slice_diagnostic(
        df,
        plane,
        0,
        rr_min,
        rr_max,
        hfit=hfit,
        pdg=pdg,
        sigma_fit_params=sigma_fit_params,
        dedx_col="dedx",
        nbins=100,
        hist_xmax=20.0,
        peak_method="kde",
        x_limits=(1.5, 4),
    )

    fname1 = f"diag_full_p{plane}_combined_rr_{rr_min:g}_{rr_max:g}.png"
    fig1.savefig(
        os.path.join(output_dir, fname1), dpi=300, bbox_inches="tight"
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, fftconvolve
from scipy.optimize import curve_fit
from scipy.stats import gaussian_kde

# ===========================================================================
# 1. Convolution and Peak Calculations
# ===========================================================================

def convolved_pdf_curve(x_grid, pdf_vals, dx, sigma):
    """Theory PDF (x) zero-mean Gaussian(sigma), evaluated back on x_grid."""
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return conv

def conv_mpv_from_grid(x_grid, conv_vals, xmin=0.0, xmax=10.0):
    """Grid-search peak location of a convolved curve, restricted to [xmin, xmax]."""
    mask = (x_grid >= xmin) & (x_grid <= xmax)
    if not np.any(mask):
        return np.nan
    idx = np.argmax(conv_vals[mask])
    return x_grid[mask][idx]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit


# ===========================================================================
# 1. Model Functions & Fit Helpers
# ===========================================================================
def exp_decay_plateau(x, a, b, c):
    """f(x) = a + b * exp(-x / c)
    Plateaus at LARGE x (used for Residual Range rr).
    """
    return a + b * np.exp(-x / c)


def exp_drop_plateau(x, a, b, c):
    """f(x) = a - b * exp(x / c)
    Plateaus at SMALL x (low MPV) and drops exponentially at LARGE x (high MPV).
    """
    return a - b * np.exp(x / c)

def fit_exp_decay(
    diag_df, y_col, x_col="rr_center", y_err_col="hist_max_err", p0=None
):
    """Fits fitting model weighted by uncertainties.
    Uses exp_decay_plateau for 'rr_center' and exp_drop_plateau for 'theory_mpv'.
    """
    xs = diag_df[x_col].to_numpy()
    ys = diag_df[y_col].to_numpy()

    has_err = bool(y_err_col and y_err_col in diag_df.columns)
    sigmas = diag_df[y_err_col].to_numpy() if has_err else None

    mask = np.isfinite(xs) & np.isfinite(ys)
    if has_err:
        mask &= np.isfinite(sigmas) & (sigmas > 0)
        sigmas = sigmas[mask]

    xs, ys = xs[mask], ys[mask]
    if len(xs) < 4:
        return None, None

    is_mpv = x_col == "theory_mpv"
    fit_func = exp_drop_plateau if is_mpv else exp_decay_plateau

    # Initial parameter estimation
    if p0 is None:
        order = np.argsort(xs)
        if is_mpv:
            # Low MPV (left) is plateau value ~ a0
            a0 = ys[order][0]
            c0 = max((xs.max() - xs.min()) / 2.0, 0.5)
            drop = abs(ys[order][0] - ys[order][-1])
            b0 = max(drop / np.exp(xs.max() / c0), 1e-4)
            p0 = [a0, b0, c0]
        else:
            # High rr (right) is plateau value ~ a0
            a0 = ys[order][-1]
            b0 = ys[order][0] - a0
            c0 = max((xs.max() - xs.min()) / 3.0, 1e-3)
            p0 = [a0, b0, c0]

    try:
        fit_kwargs = {}
        if has_err:
            fit_kwargs["sigma"] = sigmas
            fit_kwargs["absolute_sigma"] = True

        popt, pcov = curve_fit(
            fit_func, xs, ys, p0=p0, maxfev=20000, **fit_kwargs
        )
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception:
        return None, None
    

# ===========================================================================
# 2. Plotting Base Function
# ===========================================================================
def _plot_shift_diagnostics_base(
    diag_df,
    plane,
    tpc,
    x_col,
    x_label,
    fit_var_name,
    pitch=None,
    y_err_col="hist_max_err",
    out_prefix="shift_diag",
    plot_hist_conv_diff=True,
    fit_diff_hist_theory_curve=True,
    fit_diff_hist_conv_curve=True,
):
    """Core rendering function for shift diagnostics with error bars."""
    if pitch is not None:
        print(f"theoretical MPV computed at pitch = {pitch:.2f} cm")

    fig, ax = plt.subplots(figsize=(8, 5.5))
    ax.axhline(0, color="gray", lw=1, linestyle=":")

    # 1. Mode-shift from smearing
    ax.plot(
        diag_df[x_col],
        diag_df["dx_shift"],
        "o",
        color="darkorange",
        ms=4,
        label=r"conv(theory, $\sigma_G$) $-$ theory  (mode-shift from smearing)",
    )

    y_err = (
        diag_df[y_err_col]
        if (y_err_col and y_err_col in diag_df.columns)
        else None
    )

    # 2. Data Peak vs. Unsmeared Theory
    ax.errorbar(
        diag_df[x_col],
        diag_df["diff_hist_theory"],
        yerr=y_err,
        fmt="s",
        color="mediumpurple",
        ms=4,
        capsize=3,
        capthick=1,
        label=r"hist. max $-$ theory  (data peak vs. unsmeared theory)",
    )

    # 3. Data Peak vs. Convolved MPV
    if plot_hist_conv_diff and "diff_hist_conv" in diag_df.columns:
        ax.errorbar(
            diag_df[x_col],
            diag_df["diff_hist_conv"],
            yerr=y_err,
            fmt="^",
            color="teal",
            ms=4,
            capsize=3,
            capthick=1,
            label=r"hist. max $-$ conv(theory)  (data peak vs. convolved MPV)",
        )

    is_mpv = x_col == "theory_mpv"
    eval_func = exp_drop_plateau if is_mpv else exp_decay_plateau
    sign_str = "" if is_mpv else "-"

    # --- Fit 1: diff_hist_theory ---
    if fit_diff_hist_theory_curve and "diff_hist_theory" in diag_df.columns:
        popt, perr= fit_exp_decay(
            diag_df, y_col="diff_hist_theory", x_col=x_col, y_err_col=y_err_col
        )
        if popt is not None:
            a, b, c = popt
            a_e, b_e, c_e = perr
            print(
                f"diff_hist_theory fit (vs {x_col}): a={a:.4f}+/-{a_e:.4f}, "
                f"b={b:.4f}+/-{b_e:.4f}, c={c:.4f}+/-{c_e:.4f}"
            )
            xs_fit = np.linspace(
                diag_df[x_col].min(), diag_df[x_col].max(), 200
            )
            ys_fit = eval_func(xs_fit, *popt)

            lbl_op = "-" if is_mpv else "+"
            ax.plot(
                xs_fit,
                ys_fit,
                "--",
                color="mediumpurple",
                lw=1.8,
                alpha=0.8,
                label=fr"theory fit: ${a:.3f} {lbl_op} {b:.3f}\,e^{{{sign_str}{fit_var_name}/{c:.3f}}}$",
            )

    # --- Fit 2: diff_hist_conv ---
    if fit_diff_hist_conv_curve and "diff_hist_conv" in diag_df.columns:
        popt_conv, perr_conv = fit_exp_decay(
            diag_df, y_col="diff_hist_conv", x_col=x_col, y_err_col=y_err_col
        )
        if popt_conv is not None:
            a, b, c = popt_conv
            a_e, b_e, c_e = perr_conv
            print(
                f"diff_hist_conv fit (vs {x_col}):   a={a:.4f}+/-{a_e:.4f}, "
                f"b={b:.4f}+/-{b_e:.4f}, c={c:.4f}+/-{c_e:.4f}"
            )
            xs_fit = np.linspace(
                diag_df[x_col].min(), diag_df[x_col].max(), 200
            )
            ys_fit_conv = eval_func(xs_fit, *popt_conv)

            lbl_op = "-" if is_mpv else "+"
            ax.plot(
                xs_fit,
                ys_fit_conv,
                ":",
                color="teal",
                lw=2.0,
                alpha=0.9,
                label=fr"conv fit: ${a:.3f} {lbl_op} {b:.3f}\,e^{{{sign_str}{fit_var_name}/{c:.3f}}}$",
            )

    ax.set_xlabel(x_label)
    ax.set_ylabel(r"$\Delta$ MPV [MeV/cm]")
    tpc_label = "combined" if tpc == -1 else tpc
    title = f"plane {plane}, tpc {tpc_label}"
    if pitch is not None:
        title += f", pitch={pitch:.2f} cm"
    ax.set_title(title)
    ax.legend(fontsize=8, loc="best")
    ax.grid(True, linestyle=":", alpha=0.5)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_p{plane}_t{tpc}.pdf")
    plt.show()
    return fig


# Wrapper functions remain unchanged:
def plot_shift_diagnostics_vs_rr(diag_df, plane, tpc, **kwargs):
    return _plot_shift_diagnostics_base(
        diag_df=diag_df,
        plane=plane,
        tpc=tpc,
        x_col="rr_center",
        x_label="rr [cm]",
        fit_var_name="rr",
        out_prefix=kwargs.pop("out_prefix", "shift_diag_rr"),
        **kwargs,
    )


def plot_shift_diagnostics_vs_theory_mpv(diag_df, plane, tpc, **kwargs):
    return _plot_shift_diagnostics_base(
        diag_df=diag_df,
        plane=plane,
        tpc=tpc,
        x_col="theory_mpv",
        x_label=r"Theory MPV [MeV/cm]",
        fit_var_name=r"\text{MPV}",
        out_prefix=kwargs.pop("out_prefix", "shift_diag_mpv"),
        **kwargs,
    )

import numpy as np
import pandas as pd


# NOTE: robust_hist_max, kde_hist_max, local_quad_hist_max, and _PEAK_FINDERS
# are assumed to be defined/imported as in plot_slice_diagnostic_v3.py.
# Import or paste those in above this if they're not already in scope.


def compute_shift_diagnostics(
    df,
    plane,
    tpc,
    hfit,
    pdg,
    fit_params,
    dedx_col="dedx",
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    pitch=0.55,
    mass=None,
    min_entries=200,
    hist_nbins=1000,
    hist_xmin=0.0,
    hist_xmax=10.0,
    savgol_window=51,
    savgol_poly=3,
    verbose=False,
    peak_method="kde",  # "savgol" | "kde" | "local_quad" | "hybrid"
    peak_method_kwargs=None,  # extra kwargs forwarded to the chosen finder
    hybrid_rr_threshold=9.0,  # rr_center above this -> kde, below/equal -> local_quad
):
    valid_methods = set(_PEAK_FINDERS) | {"hybrid"}
    if peak_method not in valid_methods:
        raise ValueError(
            f"Unknown peak_method {peak_method!r}; choose from {sorted(valid_methods)}"
        )
    extra_kwargs = dict(peak_method_kwargs or {})

    popt, _ = fit_params.get((plane, tpc), (None, None))
    if popt is None:
        raise ValueError(f"No sigma_G(MPV) fit available for plane={plane}, tpc={tpc}")
    sl_all = df if tpc == -1 else df[df["tpc"] == tpc]
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = sl_all[(sl_all["rr"] >= lo) & (sl_all["rr"] < hi)]
        if len(sl) < min_entries:
            continue
        rr_center = 0.5 * (lo + hi)
        pdf = build_theoretical_pdf(hfit, pdg, rr_center, pitch, mass=mass)
        theory_mpv = robust_max_x_py(pdf, 0.0, 10.0, 2000)
        x_grid, pdf_vals, dx = build_pdf_grid(pdf)
        predicted_sigma_g = power_law(theory_mpv, *popt)
        conv_vals = convolved_pdf_curve(x_grid, pdf_vals, dx, predicted_sigma_g)
        conv_mpv = conv_mpv_from_grid(x_grid, conv_vals, xmin=0.0, xmax=10.0)
        dx_shift = conv_mpv - theory_mpv

        # Resolve "hybrid" to a concrete method + finder for THIS rr bin,
        # based on rr_center — mirrors plot_slice_diagnostic's dispatch.
        if peak_method == "hybrid":
            active_method = "kde" if rr_center > hybrid_rr_threshold else "local_quad"
        else:
            active_method = peak_method
        finder = _PEAK_FINDERS[active_method]

        call_kwargs = dict(extra_kwargs)
        if active_method == "savgol":
            call_kwargs.setdefault("nbins", hist_nbins)
            call_kwargs.setdefault("savgol_window", savgol_window)
            call_kwargs.setdefault("savgol_poly", savgol_poly)
        # kde / local_quad take their own defaults unless overridden via
        # peak_method_kwargs

        hist_max, hist_max_err, _, _, _ = finder(
            sl[dedx_col].to_numpy(),
            hist_xmin,
            hist_xmax,
            **call_kwargs,
        )

        diff_hist_theory = hist_max - theory_mpv
        diff_hist_conv = hist_max - conv_mpv
        if verbose:
            tag = f"hybrid\u2192{active_method}" if peak_method == "hybrid" else active_method
            print(
                f"  rr={rr_center:.2f} [{tag}]: theory_mpv={theory_mpv:.3f}, "
                f"conv_mpv={conv_mpv:.3f}, hist_max={hist_max:.3f}+/-{hist_max_err:.3f}"
            )
        rows.append(
            dict(
                plane=plane,
                tpc=tpc,
                rr_center=rr_center,
                n_hits=len(sl),
                theory_mpv=theory_mpv,
                predicted_sigma_g=predicted_sigma_g,
                conv_mpv=conv_mpv,
                dx_shift=dx_shift,
                hist_max=hist_max,
                hist_max_err=hist_max_err,
                diff_hist_theory=diff_hist_theory,
                diff_hist_conv=diff_hist_conv,
                peak_method=(
                    f"hybrid:{active_method}" if peak_method == "hybrid" else active_method
                ),
            )
        )
    return pd.DataFrame(rows)


In [ ]:
tpc = -1
plane = 2
diag_df_032 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg, fit_params=fit_params,
    pitch=0.32, verbose=False,
)
plot_shift_diagnostics_vs_rr(diag_df_032, plane, tpc, pitch=0.32, out_prefix="shift_diag_pitch032")
plot_shift_diagnostics_vs_theory_mpv(diag_df_032, plane, tpc, pitch=0.32, out_prefix="shift_diag_pitch032")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

'''
def analyze_shift_all_planes(
    hit_dfs,
    hfit,
    pdg,
    fit_params_powerlaw,  # (plane, tpc) -> power law fit params for sigma_G
    dedx_col="dedx",
    pitch=0.32,
    mass=None,
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    out_prefix="shift_diag_all",
    verbose=True,
):
    """Runs compute_shift_diagnostics for all planes and TPCs,

    fitting diff_hist_conv vs theory_mpv with exp_drop_plateau.
    """
    all_results = {}
    fit_params_shift = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[Shift Diag] Plane {plane}, TPC {tpc_label}")

            try:
                diag_df = compute_shift_diagnostics(
                    df=df,
                    plane=plane,
                    tpc=tpc,
                    hfit=hfit,
                    pdg=pdg,
                    fit_params=fit_params_powerlaw,
                    dedx_col=dedx_col,
                    rr_min=rr_min,
                    rr_max=rr_max,
                    rr_bin_width=rr_bin_width,
                    pitch=pitch,
                    mass=mass,
                    verbose=False,
                )
            except Exception as e:
                if verbose:
                    print(
                        f"  Failed compute_shift_diagnostics for ({plane}, {tpc}): {e}"
                    )
                diag_df = pd.DataFrame()

            all_results[(plane, tpc)] = diag_df

            # Fit diff_hist_conv vs theory_mpv using exp_drop_plateau
            if len(diag_df) >= 4 and "diff_hist_conv" in diag_df.columns:
                popt, perr = fit_exp_decay(
                    diag_df,
                    y_col="diff_hist_conv",
                    x_col="theory_mpv",
                    y_err_col="hist_max_err",
                )
            else:
                popt, perr = None, None

            fit_params_shift[(plane, tpc)] = (popt, perr)

    return all_results, fit_params_shift
'''

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ===========================================================================
# rr-perspective shift diagnostics: everything below fits/evaluates
# diff_hist_conv as a function of rr_center (residual range) instead of
# theory_mpv. Uses the plain fit_exp_decay (your earlier version, no
# degeneracy handling) -- for the rr axis this hasn't shown the runaway
# a/b/c cancellation that theory_mpv did, since exp_decay_plateau's
# "plateau at large x" shape matches the physical rr trend directly
# (see the working plot_shift_diagnostics_vs_rr fits: e.g.
# 0.072 - 1.115*exp(-rr/1.709)).
# ===========================================================================

PLANE_COLORS = {0: "tab:red", 1: "tab:blue", 2: "tab:green"}
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


# ---------------------------------------------------------------------------
# 1. Driver: fit diff_hist_conv vs rr_center per (plane, tpc)
# ---------------------------------------------------------------------------

def analyze_shift_all_planes(
    hit_dfs,
    hfit,
    pdg,
    fit_params_powerlaw,  # (plane, tpc) -> power law fit params for sigma_G
    dedx_col="dedx",
    pitch=0.32,
    mass=None,
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    out_prefix="shift_diag_all",
    verbose=True,
):
    """Runs compute_shift_diagnostics for all planes and TPCs, fitting
    diff_hist_conv vs rr_center with exp_decay_plateau -- the rr
    perspective, rather than theory_mpv. The correction is then read off
    directly as a function of residual range, which is what you actually
    index slices by and what you'll look up at inference time.

    Uses your plain fit_exp_decay (no degeneracy/linear-fallback
    handling) -- fit_params_shift[(plane, tpc)] is always either
    (popt, perr) for a 3-parameter exp_decay_plateau fit, or (None, None)
    if there weren't enough points or the fit failed.
    """
    all_results = {}
    fit_params_shift = {}
    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[Shift Diag] Plane {plane}, TPC {tpc_label}")
            try:
                diag_df = compute_shift_diagnostics(
                    df=df,
                    plane=plane,
                    tpc=tpc,
                    hfit=hfit,
                    pdg=pdg,
                    fit_params=fit_params_powerlaw,
                    dedx_col=dedx_col,
                    rr_min=rr_min,
                    rr_max=rr_max,
                    rr_bin_width=rr_bin_width,
                    pitch=pitch,
                    mass=mass,
                    verbose=False,
                )
            except Exception as e:
                if verbose:
                    print(
                        f"  Failed compute_shift_diagnostics for ({plane}, {tpc}): {e}"
                    )
                diag_df = pd.DataFrame()
            all_results[(plane, tpc)] = diag_df

            # == Fit diff_hist_conv vs rr_center (not theory_mpv) with
            # == exp_decay_plateau -- plateaus at LARGE rr
            if len(diag_df) >= 4 and "diff_hist_conv" in diag_df.columns:
                popt, perr = fit_exp_decay(
                    diag_df,
                    y_col="diff_hist_conv",
                    x_col="rr_center",
                    y_err_col="hist_max_err",
                )
            else:
                popt, perr = None, None
            fit_params_shift[(plane, tpc)] = (popt, perr)
    return all_results, fit_params_shift


# ---------------------------------------------------------------------------
# 2. Eval/label helpers -- rr perspective only, always exp_decay_plateau
# ---------------------------------------------------------------------------

def eval_shift_fit_rr(rr, popt):
    """Evaluates the rr-perspective fit -- always exp_decay_plateau,
    since analyze_shift_all_planes now only ever produces that shape
    (no linear fallback: this uses your plain fit_exp_decay)."""
    return exp_decay_plateau(rr, *popt)


def format_fit_label_rr(prefix, popt):
    a, b, c = popt
    return f"{prefix}: {a:.3f} + {b:.3f}e^{{-rr/{c:.3f}}}"


# ---------------------------------------------------------------------------
# 3. All planes/TPCs on one square plot, vs rr
# ---------------------------------------------------------------------------


In [ ]:
# ===========================================================================
# Shift Diagnostics Execution Workflow
# ===========================================================================

# 1. Run shift diagnostics analysis across all planes & TPCs
#    (Passes sigma_fit_params obtained from analyze_theoretical)
all_shift_results, shift_fit_params = analyze_shift_all_planes(
    hit_dfs,
    hfit,
    pdg,
    fit_params_powerlaw=sigma_fit_params,
    dedx_col="dedx",
    pitch=0.32,
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    out_prefix="shift_diag",
    verbose=True,
)



In [ ]:

def plot_all_planes_shift(
    all_results, fit_params_shift, out_prefix="shift_diag"
):
    """Plots all (plane, tpc) exp_decay_plateau fit curves for
    diff_hist_conv vs rr_center on a single square plot."""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.axhline(0, color="gray", lw=1, linestyle=":")

    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params_shift.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS.get(plane, "tab:blue")
        style_info = TPC_STYLES.get(
            tpc, {"linestyle": "-", "label_prefix": f"tpc={tpc}"}
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["rr_center"].min(), res["rr_center"].max(), 200)
        ys = eval_shift_fit_rr(xs, popt)
        label = format_fit_label_rr(f"{prefix}, Plane {plane}", popt)

        ax.plot(xs, ys, linestyle=style, color=color, linewidth=2.0, label=label)

    ax.set_xlabel("rr [cm]", fontsize=12)
    ax.set_ylabel(
        r"$\Delta$ MPV (Hist Max $-$ Conv Theory) [MeV/cm]", fontsize=12
    )
    ax.legend(fontsize=9, loc="best", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


# ---------------------------------------------------------------------------
# 4. One subplot per plane, vs rr (full range or zoomed via x_limits)
# ---------------------------------------------------------------------------

def plot_by_plane_shift(
    all_results,
    fit_params_shift,
    out_prefix="shift_diag",
    x_limits=None,   # now an rr range, e.g. [1.9, 3.0] cm -- not an MPV range
):
    """One square subplot per plane displaying data points and
    exp_decay_plateau fits for diff_hist_conv vs rr_center. x_limits, if
    given, is an rr window in cm."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        ax.axhline(0, color="gray", lw=1, linestyle=":")
        y_visible_min, y_visible_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt, _ = fit_params_shift.get((plane, tpc), (None, None))

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]
            style = style_info["linestyle"]
            marker = style_info["marker"]
            prefix = style_info["label_prefix"]

            if x_limits is not None:
                mask = (res["rr_center"] >= x_limits[0]) & (
                    res["rr_center"] <= x_limits[1]
                )
                res_filtered = res[mask]
            else:
                res_filtered = res

            if len(res_filtered):
                ax.errorbar(
                    res_filtered["rr_center"],
                    res_filtered["diff_hist_conv"],
                    yerr=res_filtered["hist_max_err"],
                    fmt=marker,
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{prefix} Data",
                )
                if x_limits is not None:
                    y_visible_min = min(
                        y_visible_min,
                        (
                            res_filtered["diff_hist_conv"]
                            - res_filtered["hist_max_err"]
                        ).min(),
                    )
                    y_visible_max = max(
                        y_visible_max,
                        (
                            res_filtered["diff_hist_conv"]
                            + res_filtered["hist_max_err"]
                        ).max(),
                    )

            if popt is not None:
                x_min = x_limits[0] if x_limits else res["rr_center"].min()
                x_max = x_limits[1] if x_limits else res["rr_center"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = eval_shift_fit_rr(xs, popt)
                label_fit = format_fit_label_rr(f"{prefix} Fit", popt)

                ax.plot(
                    xs, ys, linestyle=style, color=color, linewidth=1.8,
                    label=label_fit,
                )

                if x_limits is not None:
                    y_visible_min = min(y_visible_min, ys.min())
                    y_visible_max = max(y_visible_max, ys.max())

        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_visible_min) and not np.isinf(y_visible_max):
                y_pad = (y_visible_max - y_visible_min) * 0.1
                ax.set_ylim(y_visible_min - y_pad, y_visible_max + y_pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("rr [cm]", fontsize=12)
        ax.set_ylabel(
            r"$\Delta$ MPV (Hist Max $-$ Conv Theory) [MeV/cm]", fontsize=12
        )
        ax.legend(fontsize=8, loc="best", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_by_plane{zoom_suffix}.pdf")
    plt.show()


# ---------------------------------------------------------------------------
# 5. Single-slice sanity check, shift now predicted from rr_center
# ---------------------------------------------------------------------------

    return fig

'''
import matplotlib.pyplot as plt
import numpy as np

# Plane color mapping for plot_all_planes_shift
PLANE_COLORS = {0: "tab:red", 1: "tab:blue", 2: "tab:green"}

# Style mapping for consistency across all functions
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}

def eval_shift_fit(xs, popt, kind, is_mpv):
    """Evaluates a fit produced by fit_exp_decay_robust, dispatching on
    `kind` rather than assuming a fixed parameter count -- 'exp' fits use
    exp_drop_plateau/exp_decay_plateau (3 params), 'linear' fits use a
    plain line (2 params, from np.polyfit's [slope, intercept] order)."""
    if kind == "linear":
        m, b0 = popt
        return m * xs + b0
    eval_func = exp_drop_plateau if is_mpv else exp_decay_plateau
    return eval_func(xs, *popt)


def format_fit_label(prefix, popt, kind, is_mpv):
    if kind == "linear":
        m, b0 = popt
        return f"{prefix}: {b0:.3f} + {m:.3f}\u00b7x  (linear)"
    a, b, c = popt
    op = "-" if is_mpv else "+"
    var = "x" if is_mpv else "rr"
    return f"{prefix}: {a:.3f} {op} {b:.3f}e^{{{var}/{c:.3f}}}"
    
def plot_all_planes_shift(
    all_results, fit_params_shift, out_prefix="shift_diag"
):
    """Plots all (plane, tpc) fit curves for diff_hist_conv on a single
    square plot. Handles both 'exp' and 'linear' fit kinds transparently
    via eval_shift_fit."""
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.axhline(0, color="gray", lw=1, linestyle=":")

    for (plane, tpc), res in all_results.items():
        entry = fit_params_shift.get((plane, tpc), (None, None, None))
        # == tolerate old-style 2-tuples still lingering in a dict, just
        # == in case not everything downstream was regenerated
        if len(entry) == 3:
            popt, _, kind = entry
        else:
            popt, _ = entry
            kind = "exp" if popt is not None and len(popt) == 3 else None

        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS.get(plane, "tab:blue")
        style_info = TPC_STYLES.get(
            tpc, {"linestyle": "-", "label_prefix": f"tpc={tpc}"}
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["theory_mpv"].min(), res["theory_mpv"].max(), 200)
        ys = eval_shift_fit(xs, popt, kind, is_mpv=True)
        label = format_fit_label(f"{prefix}, Plane {plane}", popt, kind, is_mpv=True)

        ax.plot(xs, ys, linestyle=style, color=color, linewidth=2.0, label=label)

    ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
    ax.set_ylabel(r"$\Delta$ MPV (Hist Max $-$ Conv Theory) [MeV/cm]", fontsize=12)
    ax.legend(fontsize=9, loc="best", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()
'''

In [ ]:
shift_fit_params

In [ ]:
# 2. Plot exp_drop_plateau fit curves across all planes on a single plot
plot_all_planes_shift(
    all_shift_results,
    shift_fit_params,
    out_prefix="shift_diag_comparison",
)

# 3. Plot shift diagnostics by plane (full MPV range)
plot_by_plane_shift(
    all_shift_results,
    shift_fit_params,
    out_prefix="shift_diag_comparison",
)

# 4. Plot shift diagnostics by plane (zoomed MPV range)
plot_by_plane_shift(
    all_shift_results,
    shift_fit_params,
    out_prefix="shift_diag_comparison",
    x_limits=[1.9, 3.0],
)

In [ ]:

def plot_slice_diagnostic_shiftfit(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    sigma_fit_params=None,
    shift_fit_params=None,      # now keyed to an rr-fit: exp_decay_plateau(rr_center, *popt)
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    x_limits=None,
    shift_func=None,            # defaults to exp_decay_plateau, evaluated at rr_center
    shift_param_index=0,        # index into shift_fit_params[(plane, tpc)] to grab popt
    show_measured_peak=True,    # overlay the empirically-measured peak for comparison only
    peak_method="hybrid",       # only used if show_measured_peak=True
    peak_method_kwargs=None,
    hybrid_rr_threshold=9.0,
    savgol_window=51,
    savgol_poly=3,
    peak_nbins=500,
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice, where
    the convolved theory curve is shifted using the GLOBALLY-FITTED shift
    model from analyze_shift_all_planes -- now a function of rr_center
    (residual range) rather than theory_mpv, matching the rr-perspective
    fit above.

    shift_func defaults to exp_decay_plateau and is evaluated at this
    slice's rr_center (not theo_mpv) -- pass a different callable only if
    you've fit shift_fit_params with something else.

    If show_measured_peak=True, the empirically-measured data peak is
    still drawn (using peak_method) purely for visual comparison -- it
    does NOT feed into the shift calculation.
    """
    if shift_func is None:
        shift_func = exp_decay_plateau

    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]
    rr_center = 0.5 * (rr_min + rr_max)

    fig, ax = plt.subplots(figsize=(9, 6))

    # --- Stage 1: Data Histogram ---
    data_vals = sl[dedx_col].dropna().to_numpy()
    counts, edges, _ = ax.hist(
        data_vals,
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"Data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 500)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])

    data_max = counts.max() if len(counts) > 0 else 0.0
    curve_max = data_max

    # 1. (Optional) Measured Data Peak Line -- comparison only, not used below
    if show_measured_peak and len(data_vals) > 0 and data_max > 0:
        try:
            active_method = (
                ("kde" if rr_center > hybrid_rr_threshold else "local_quad")
                if peak_method == "hybrid"
                else peak_method
            )
            finder = _PEAK_FINDERS[active_method]
            call_kwargs = dict(peak_method_kwargs or {})
            if active_method == "savgol":
                call_kwargs.setdefault("nbins", peak_nbins)
                call_kwargs.setdefault("savgol_window", savgol_window)
                call_kwargs.setdefault("savgol_poly", savgol_poly)
            measured_peak_x, _, _, _, _ = finder(
                data_vals, hist_xmin, hist_xmax, **call_kwargs
            )
        except Exception:
            measured_peak_x = bin_centers[np.argmax(counts)]

        ax.axvline(
            measured_peak_x,
            color="navy",
            linestyle=":",
            linewidth=1.8,
            label=f"Measured Peak (ref only, {measured_peak_x:.2f})",
        )

    if hfit is not None:
        mean_pitch = 0.32
        pdf = build_theoretical_pdf(
            hfit, pdg, rr_center, mean_pitch, mass=mass
        )

        # --- Stage 2: Theory PDF (Unconvolved) ---
        theo_mpv = pdf.GetMaximumX()
        y_theo = np.array([pdf.Eval(xv) for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = (y_theo / y_theo.max()) * data_max

        ax.plot(
            x_eval, y_theo, color="red", linestyle=":", linewidth=2.5,
            label="1. Theory Unconvolved",
        )
        ax.axvline(
            theo_mpv, color="red", linestyle="--", linewidth=1.5, alpha=0.8,
            label=f"Theory MPV ({theo_mpv:.2f})",
        )
        curve_max = max(curve_max, y_theo.max())

        popt_sig = (
            sigma_fit_params.get((plane, tpc), (None, None))[0]
            if sigma_fit_params else None
        )

        # --- Stage 3: Convolved PDF (Unshifted) ---
        if popt_sig is not None:
            predicted_sigma_g = power_law(theo_mpv, *popt_sig)
            y_pred_unsh = convolve_pdf_with_gaussian(
                pdf, predicted_sigma_g, x_eval, recenter_to_mode=False
            )
            if y_pred_unsh.max() > 0 and data_max > 0:
                y_pred_unsh = (y_pred_unsh / y_pred_unsh.max()) * data_max

            unsh_peak_x = x_eval[np.argmax(y_pred_unsh)]

            ax.plot(
                x_eval, y_pred_unsh, color="mediumpurple", linestyle="--",
                linewidth=2.0, label=r"2. Theory $\otimes$ Res (Unshifted)",
            )
            ax.axvline(
                unsh_peak_x, color="mediumpurple", linestyle="-.",
                linewidth=1.5, alpha=0.8,
                label=f"Unshifted Peak ({unsh_peak_x:.2f})",
            )
            curve_max = max(curve_max, y_pred_unsh.max())

            # --- Stage 4: Convolved PDF shifted using the rr-FITTED shift model ---
            shift_entry = (
                shift_fit_params.get((plane, tpc), None) if shift_fit_params else None
            )
            popt_shift = (
                shift_entry[shift_param_index] if shift_entry is not None else None
            )
            if popt_shift is not None:
                # == evaluated at rr_center, not theo_mpv -- this is the
                # == rr-perspective correction from analyze_shift_all_planes
                predicted_shift = shift_func(rr_center, *popt_shift)

                y_pred_shifted = np.interp(
                    x_eval - predicted_shift, x_eval, y_pred_unsh
                )
                # Exact by construction -- avoid re-deriving via argmax on
                # x_eval, which reintroduces grid-quantization error.
                shf_peak_x = unsh_peak_x + predicted_shift

                ax.plot(
                    x_eval, y_pred_shifted, color="darkorange", linestyle="-",
                    linewidth=2.5,
                    label=r"3. Theory $\otimes$ Res (Fit-Predicted Shift, vs rr)",
                )
                ax.axvline(
                    shf_peak_x, color="darkorange", linestyle="-",
                    linewidth=1.5, alpha=0.8,
                    label=f"Predicted Peak ({shf_peak_x:.2f}, "
                          f"\u0394={predicted_shift:+.3f} @ rr={rr_center:.1f})",
                )
                curve_max = max(curve_max, y_pred_shifted.max())

    # --- Zoom Limits & Dynamic Y-Rescaling ---
    if x_limits is not None:
        ax.set_xlim(x_limits)
        mask = (x_eval >= x_limits[0]) & (x_eval <= x_limits[1])

        visible_max = data_max
        if "y_theo" in locals() and len(y_theo[mask]):
            visible_max = max(visible_max, y_theo[mask].max())
        if "y_pred_unsh" in locals() and len(y_pred_unsh[mask]):
            visible_max = max(visible_max, y_pred_unsh[mask].max())
        if "y_pred_shifted" in locals() and len(y_pred_shifted[mask]):
            visible_max = max(visible_max, y_pred_shifted[mask].max())

        ax.set_ylim(0, visible_max * 1.15 if visible_max > 0 else 1.0)
    else:
        ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    ax.set_xlabel(r"$dE/dx$ [MeV/cm]", fontsize=12)
    ax.set_ylabel(f"Hits / {bin_width:.2f} MeV/cm", fontsize=12)

    tpc_title = "TPCs Combined" if tpc == -1 else f"TPC {tpc}"
    ax.set_title(
        f"Plane {plane}, {tpc_title}, {rr_min:g} <= RR < {rr_max:g} cm "
        f"[shift_fit vs rr]",
        fontsize=13,
    )

    ax.legend(fontsize=8, loc="upper right", framealpha=0.9, ncol=2)
    ax.grid(True, linestyle=":", alpha=0.5)
    fig.tight_layout()

    return fig

In [ ]:
import os


output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(rr, rr + 1) for rr in range(1, 40)]  # (1,2), (2,3), ..., (39,40)
plane = 2
df = hit_dfs[plane]
for rr_min, rr_max in rr_ranges:
    fig1 = plot_slice_diagnostic_shiftfit(
        df,
        plane,
        -1,
        rr_min,
        rr_max,
        hfit=hfit,
        pdg=pdg,
        sigma_fit_params=sigma_fit_params,
        shift_fit_params=shift_fit_params,   # from analyze_shift_all_planes
        dedx_col="dedx",
        nbins=100,
        hist_xmax=20.0,
        x_limits=(1.5, 4),
    )
    fname1 = f"diag_shiftfit_p{plane}_combined_rr_{rr_min:g}_{rr_max:g}.png"
    fig1.savefig(
        os.path.join(output_dir, fname1), dpi=300, bbox_inches="tight"
    )

In [ ]:
def plot_shift_fit_diagnostic(
    df,
    plane,
    tpc,
    hfit,
    pdg,
    sigma_fit_params,
    shift_fit_params,
    dedx_col="dedx",
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    pitch=0.32,
    mass=None,
    min_entries=200,
    hist_nbins=1000,
    hist_xmin=0.0,
    hist_xmax=10.0,
    savgol_window=51,
    savgol_poly=3,
    peak_method="hybrid",
    peak_method_kwargs=None,
    hybrid_rr_threshold=9.0,
    shift_func=None,          # defaults to exp_decay_plateau, evaluated at rr_center
    shift_param_index=0,      # index into shift_fit_params[(plane, tpc)]
    show_residuals=True,
    figsize=(9, 7),
):
    """Compares the globally-fitted shift model (shift_fit_params, from
    analyze_shift_all_planes) against the empirically-measured shift
    (data histogram peak minus theory), across rr, for one (plane, tpc).

    Top panel: measured shift (points w/ bootstrap error bars) vs the
    fitted prediction curve (line), both vs rr_center.
    Bottom panel (optional): residual = measured - predicted, in absolute
    MeV/cm (same units as the top panel).

    predicted_shift is evaluated at rr_centers, matching what
    analyze_shift_all_planes actually fit shift_fit_params against (not
    theory_mpv) -- evaluating it anywhere else would plot the fitted
    curve at the wrong x-values entirely.

    Uses compute_shift_diagnostics(..., peak_method=peak_method) to get
    the measured side per rr bin -- make sure that function is in scope
    with its peak_method/hybrid_rr_threshold support (compute_shift_diagnostics_v2).
    """
    if shift_func is None:
        shift_func = exp_decay_plateau

    popt_shift_entry = shift_fit_params.get((plane, tpc), None) if shift_fit_params else None
    if popt_shift_entry is None:
        raise ValueError(f"No shift fit available for plane={plane}, tpc={tpc}")
    popt_shift = popt_shift_entry[shift_param_index]
    if popt_shift is None:
        raise ValueError(f"Shift fit for plane={plane}, tpc={tpc} is None (fit failed or too few points)")

    diag_df = compute_shift_diagnostics(
        df,
        plane,
        tpc,
        hfit,
        pdg,
        sigma_fit_params,
        dedx_col=dedx_col,
        rr_min=rr_min,
        rr_max=rr_max,
        rr_bin_width=rr_bin_width,
        pitch=pitch,
        mass=mass,
        min_entries=min_entries,
        hist_nbins=hist_nbins,
        hist_xmin=hist_xmin,
        hist_xmax=hist_xmax,
        savgol_window=savgol_window,
        savgol_poly=savgol_poly,
        peak_method=peak_method,
        peak_method_kwargs=peak_method_kwargs,
        hybrid_rr_threshold=hybrid_rr_threshold,
    )

    if diag_df.empty:
        raise ValueError(
            f"compute_shift_diagnostics returned no rows for plane={plane}, "
            f"tpc={tpc} (check min_entries / rr range)."
        )

    # "True"/measured shift: data histogram peak vs the convolved theory
    # peak -- this is what shift_fit_params should have been fit against.
    measured_shift = diag_df["diff_hist_conv"].to_numpy()
    measured_shift_err = diag_df["hist_max_err"].to_numpy()
    rr_centers = diag_df["rr_center"].to_numpy()

    # == evaluate at rr_centers, matching what analyze_shift_all_planes
    # == actually fit against -- not theory_mpvs
    predicted_shift = shift_func(rr_centers, *popt_shift)

    if show_residuals:
        fig, (ax_main, ax_res) = plt.subplots(
            2, 1, figsize=figsize, sharex=True,
            gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08},
        )
    else:
        fig, ax_main = plt.subplots(figsize=figsize)
        ax_res = None

    ax_main.errorbar(
        rr_centers, measured_shift, yerr=measured_shift_err,
        fmt="o", color="navy", markersize=5, capsize=3,
        label="Measured shift (data peak $-$ conv. theory peak)",
    )
    order = np.argsort(rr_centers)
    ax_main.plot(
        rr_centers[order], predicted_shift[order],
        color="darkorange", linewidth=2.0,
        label="Fitted prediction (vs rr)",
    )
    ax_main.axhline(0.0, color="gray", linewidth=0.8, linestyle=":")
    ax_main.set_ylabel(r"Shift [MeV/cm]", fontsize=12)
    tpc_title = "TPCs Combined" if tpc == -1 else f"TPC {tpc}"
    ax_main.set_title(
        f"Plane {plane}, {tpc_title}: Measured vs Fitted dE/dx Peak Shift",
        fontsize=13,
    )
    ax_main.legend(fontsize=9, loc="best")
    ax_main.grid(True, linestyle=":", alpha=0.5)

    if show_residuals:
        residuals = measured_shift - predicted_shift
        ax_res.errorbar(
            rr_centers, residuals, yerr=measured_shift_err,
            fmt="o", color="navy", markersize=4, capsize=2,
        )
        ax_res.axhline(0.0, color="darkorange", linewidth=1.5)
        ax_res.set_ylabel("Residual\n(meas.$-$pred.) [MeV/cm]", fontsize=10)
        ax_res.set_xlabel(r"Residual Range RR [cm]", fontsize=12)
        ax_res.grid(True, linestyle=":", alpha=0.5)
    else:
        ax_main.set_xlabel(r"Residual Range RR [cm]", fontsize=12)

    fig.tight_layout()
    return fig, diag_df

In [ ]:

fig, diag_df = plot_shift_fit_diagnostic(
    hit_dfs[2],
    plane=2,
    tpc=0,
    hfit=hfit,
    pdg=pdg,
    sigma_fit_params=sigma_fit_params,
    shift_fit_params=shift_fit_params,
    rr_min=1.0,
    rr_max=40.0,
    rr_bin_width=1.0,
)
fig.savefig(
    os.path.join(output_dir, "shift_fit_vs_measured_p2_tpc0.png"),
    dpi=300, bbox_inches="tight",
)


In [ ]:
print(pdg)

In [ ]:
def build_shift_map_cpp(shift_fit_params_by_pdg, tpc=-1, n_planes=3, var_name="pdg_plane_shift_map", verbose=True):
    """Builds the C++ initializer for PhysdEdx::pdg_plane_shift_map from
    one or more analyze_shift_all_planes runs.

    shift_fit_params_by_pdg: {pdg: shift_fit_params}. Keys MUST be the
    literal pdg codes (13, 211, 2212) -- not a variable that happens to
    hold one of those values at call time. A stale/reassigned `pdg`
    variable used as a dict key here will silently mislabel which
    species's fit ends up under which pdg, or drop it to the -1
    placeholder if the lookup misses entirely -- exactly the kind of
    mismatch that makes the printed map disagree with what
    plot_all_planes_shift shows for the same shift_fit_params.

    tpc: which TPC's fit to pull (-1 = combined, matching
    get_conv_function_map's usage).

    verbose: if True, prints a per-(plane, pdg) summary of what was
    actually used, in the same "a + b*e^{-rr/c}" format as the plot
    legends, so you can directly compare against the figure and catch
    mismatches immediately rather than downstream in ROOT.
    """
    plane_names = [f"Plane {i}" for i in range(n_planes)]
    pdg_order = [13, 2212, 211]  # muon, proton, pion -- matches pdg_plane_map's existing order

    if verbose:
        bad_keys = [k for k in shift_fit_params_by_pdg if k not in pdg_order]
        if bad_keys:
            print(f"[build_shift_map_cpp] WARNING: shift_fit_params_by_pdg has "
                  f"unexpected keys {bad_keys} -- expected a subset of {pdg_order}. "
                  f"This usually means a variable (e.g. `pdg`) was used as a dict "
                  f"key instead of a literal pdg code.")

    lines = [f"vector<std::map<int, vector<double>>> PhysdEdx::{var_name} = {{"]
    for plane in range(n_planes):
        lines.append(f"  // {plane_names[plane]}")
        lines.append("  {")
        entry_lines = []
        for pdg in pdg_order:
            popt = None
            fit_params = shift_fit_params_by_pdg.get(pdg)
            if fit_params is not None:
                entry = fit_params.get((plane, tpc))
                if entry is not None:
                    popt = entry[0]
            a, b, c = (popt if popt is not None else (-1.0, -1.0, -1.0))
            entry_lines.append(f"    {{{pdg}, {{{a:.8g}, {b:.8g}, {c:.8g}}}}}")

            if verbose:
                if popt is None:
                    print(f"  plane={plane}, pdg={pdg}, tpc={tpc}: "
                          f"NO FIT -> {{-1, -1, -1}}")
                else:
                    print(f"  plane={plane}, pdg={pdg}, tpc={tpc}: "
                          f"{a:.3f} + {b:.3f}e^{{-rr/{c:.3f}}}")
        lines.append(",\n".join(entry_lines))
        lines.append("  }" + ("," if plane < n_planes - 1 else ""))
    lines.append("};")
    return "\n".join(lines)

shift_fit_params_by_pdg = {
    211: shift_fit_params,   # muon -- from analyze_shift_all_planes(hit_dfs, hfit, pdg=13, ...)
    # 211: shift_fit_params_pion,     # add if/when you run it for pions
    # 2212: shift_fit_params_proton,  # add if/when you run it for protons
}
print(build_shift_map_cpp(shift_fit_params_by_pdg, tpc=-1))

In [ ]:
import numpy as np


def evaluate_and_print_shifts(shift_fit_params, rr_slices=None):
    """Evaluates the exponential shift model for each (plane, tpc) pair

    across given residual range (rr) values and prints the results.
    """
    if rr_slices is None:
        # Default half-integer rr slice centers (e.g., rr in range 1-2 cm has center 1.5)
        rr_slices = [0.5, 1.5, 2.5, 3.5, 4.5, 6.5, 15.5, 30.5]

    def exp_decay_plateau(rr, p0, p1, p2):
        return p0 + p1 * np.exp(-rr / p2)

    header = (
        f"{'Plane':<6} | {'TPC':<5} | {'rr [cm]':<8} | {'dx (Shift) [MeV/cm]'}"
    )
    print(header)
    print("-" * len(header))

    for (plane, tpc), fit_entry in sorted(shift_fit_params.items()):
        if fit_entry is None or fit_entry[0] is None:
            continue

        popt = fit_entry[0]  # Grab [p0, p1, p2] array

        # Skip invalid/unfit parameters
        if np.all(popt == -1):
            print(f"{plane:<6} | {tpc:<5} | {'N/A':<8} | Shift fit unassigned (-1)")
            print("-" * len(header))
            continue

        for rr in rr_slices:
            dx = exp_decay_plateau(rr, *popt)
            print(f"{plane:<6} | {tpc:<5} | {rr:<8.1f} | {dx:+.6f}")

        print("-" * len(header))


# Run with your existing shift_fit_params dictionary:
evaluate_and_print_shifts(shift_fit_params)